# 🧠 Research Paper Intelligence Engine (Standalone Colab Version)

This notebook is completely self-contained. Run all cells from top to bottom. It will write the source code to the Colab environment, install dependencies, and launch the Streamlit web app.

In [ ]:
!mkdir -p src
!mkdir -p data
!mkdir -p documents
!mkdir -p embeddings
!mkdir -p vector_store

In [ ]:
%%writefile requirements.txt
# ============================================================
# Research Paper Intelligence Engine — RAG System
# Requirements
# ============================================================

# PDF Processing
PyMuPDF>=1.23.0

# Text Splitting & LLM Chains
langchain>=0.1.0
langchain-community>=0.0.20

# Embeddings
sentence-transformers>=2.2.2

# Vector Store
faiss-cpu>=1.7.4

# HuggingFace Models (Summarization / QA)
transformers>=4.36.0
torch>=2.0.0

# UI
streamlit>=1.30.0

# Utilities
numpy>=1.24.0
tqdm>=4.66.0

# Agentic Workflow
langgraph>=0.0.20
langchain-core>=0.1.0
arxiv>=2.1.0
pydantic>=2.0.0

# PDF Export
fpdf2>=2.7.0


In [ ]:
%%writefile config.py
# ============================================================
# config.py — Central Configuration
# All tunable parameters live here so the rest of the code
# never needs magic numbers.
# ============================================================

import os

# ── Paths ────────────────────────────────────────────────────
BASE_DIR        = os.path.dirname(os.path.abspath(__file__))
DATA_DIR        = os.path.join(BASE_DIR, "data")
DOCUMENTS_DIR   = os.path.join(BASE_DIR, "documents")
EMBEDDINGS_DIR  = os.path.join(BASE_DIR, "embeddings")
VECTOR_STORE_DIR= os.path.join(BASE_DIR, "vector_store")

# Create directories if they don't exist
for _dir in [DATA_DIR, DOCUMENTS_DIR, EMBEDDINGS_DIR, VECTOR_STORE_DIR]:
    os.makedirs(_dir, exist_ok=True)

# ── Chunking ─────────────────────────────────────────────────
CHUNK_SIZE      = 500       # characters per chunk
CHUNK_OVERLAP   = 100       # overlap between consecutive chunks

# ── Embedding Model ──────────────────────────────────────────
# A lightweight, high-quality model that runs well on CPU
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ── FAISS ────────────────────────────────────────────────────
FAISS_INDEX_FILE = os.path.join(VECTOR_STORE_DIR, "faiss_index.index")
CHUNKS_META_FILE = os.path.join(VECTOR_STORE_DIR, "chunks_meta.json")

# ── Retrieval ────────────────────────────────────────────────
TOP_K_RESULTS   = 5         # number of chunks to retrieve per query

# ── Summarization / QA Model ────────────────────────────────
# facebook/bart-large-cnn is fine for CPU; swap for a larger
# model if running on GPU/Colab A100.
SUMMARIZATION_MODEL = "facebook/bart-large-cnn"
QA_MODEL            = "google/flan-t5-large"

# ── Agentic Workflow ─────────────────────────────────────────
ARXIV_SEARCH_LIMIT  = 10        # Max papers to fetch from arXiv
AGENT_TOP_K         = 3         # Top K most relevant papers to analyze

# ── Logging ──────────────────────────────────────────────────
LOG_LEVEL = "INFO"


In [ ]:
%%writefile src/pdf_processor.py
# ============================================================
# src/pdf_processor.py
# Phase 1 — PDF Processing
#
# Objective:
#   Extract raw text from one or many PDF files, clean it,
#   and save plain-text versions to the documents/ folder.
#
# Architecture:
#   PDFProcessor class
#     ├── extract_text(pdf_path) -> str
#     ├── clean_text(raw_text)   -> str
#     └── process_all(pdf_dir)  -> dict[filename, clean_text]
#
# Dependencies: PyMuPDF (fitz), os, logging, re
# ============================================================

import os
import re
import logging
import unicodedata
import fitz  # PyMuPDF

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import DATA_DIR, DOCUMENTS_DIR, LOG_LEVEL

# ── Logger setup ─────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class PDFProcessor:
    """
    Handles extraction and cleaning of text from PDF research papers.

    Usage:
        processor = PDFProcessor()
        texts = processor.process_all()   # reads from data/
        # or process a single file:
        text = processor.extract_text("path/to/paper.pdf")
    """

    def __init__(self, data_dir: str = DATA_DIR, docs_dir: str = DOCUMENTS_DIR):
        self.data_dir = data_dir
        self.docs_dir = docs_dir
        os.makedirs(self.docs_dir, exist_ok=True)
        logger.info("PDFProcessor initialised.")
        logger.info(f"  Input  dir : {self.data_dir}")
        logger.info(f"  Output dir : {self.docs_dir}")

    # ── Core extraction ──────────────────────────────────────

    def extract_text(self, pdf_path: str = None, stream: bytes = None) -> str:
        """
        Extract raw text from a single PDF file page by page.

        Args:
            pdf_path: Absolute or relative path to the PDF (optional).
            stream: Raw bytes of the PDF file (optional).

        Returns:
            A single string containing all pages concatenated.
        """
        if stream is not None:
            logger.info("Extracting text from memory stream")
            try:
                doc = fitz.open(stream=stream, filetype="pdf")
            except Exception as exc:
                raise RuntimeError(f"Failed to open PDF from stream: {exc}") from exc
        elif pdf_path is not None:
            if not os.path.isfile(pdf_path):
                raise FileNotFoundError(f"PDF not found: {pdf_path}")
            logger.info(f"Extracting text from: {os.path.basename(pdf_path)}")
            try:
                doc = fitz.open(pdf_path)
            except Exception as exc:
                raise RuntimeError(f"Failed to open PDF '{pdf_path}': {exc}") from exc
        else:
            raise ValueError("Must provide either pdf_path or stream")

        pages_text = []
        try:
            for page in doc:
                pages_text.append(page.get_text("text"))
            doc.close()
        except Exception as exc:
            raise RuntimeError(f"Error during extraction: {exc}") from exc

        raw_text = "\n".join(pages_text)
        logger.info(f"  Extracted {len(raw_text):,} chars.")
        return raw_text

    # ── Text cleaning ────────────────────────────────────────

    def clean_text(self, raw_text: str) -> str:
        """
        Remove artefacts common in PDF-extracted text:
          - Excessive whitespace / blank lines
          - Hyphenated line breaks (re-join words)
          - Non-ASCII junk characters
          - Page headers/footers patterns

        Args:
            raw_text: The raw string from extract_text().

        Returns:
            A cleaner, more readable string.
        """
        text = raw_text

        # Normalize Unicode ligatures (e.g. \ufb01 -> fi, \ufb02 -> fl)
        text = unicodedata.normalize("NFKD", text)

        # Re-join hyphenated words split across lines (common in PDFs)
        text = re.sub(r"-\n", "", text)

        # Remove excessive newlines (3+ → 2)
        text = re.sub(r"\n{3,}", "\n\n", text)

        # Replace non-breaking spaces and tabs with regular space
        text = text.replace("\xa0", " ").replace("\t", " ")

        # Remove lines that are just page numbers (e.g. "— 3 —" or just "3")
        text = re.sub(r"^\s*[\-–—]?\s*\d+\s*[\-–—]?\s*$", "", text, flags=re.MULTILINE)

        # Collapse multiple spaces into one
        text = re.sub(r" {2,}", " ", text)

        # Strip leading/trailing whitespace from each line
        lines = [line.strip() for line in text.splitlines()]
        text = "\n".join(lines)

        # Final strip
        text = text.strip()

        logger.debug(f"  Cleaned text length: {len(text):,} characters.")
        return text

    # ── Save to disk ─────────────────────────────────────────

    def save_text(self, filename: str, text: str) -> str:
        """
        Save cleaned text to documents/ folder as a .txt file.

        Args:
            filename: Base name (e.g. 'paper1.pdf' → 'paper1.txt').
            text: The cleaned text string.

        Returns:
            Path to the saved .txt file.
        """
        base = os.path.splitext(filename)[0]
        out_path = os.path.join(self.docs_dir, f"{base}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
        logger.info(f"  Saved cleaned text → {out_path}")
        return out_path

    # ── Batch processing ─────────────────────────────────────

    def process_all(self) -> dict:
        """
        Process every PDF in data_dir.

        Returns:
            {
                "paper1.pdf": {
                    "raw_text": "...",
                    "clean_text": "...",
                    "txt_path": "documents/paper1.txt"
                },
                ...
            }
        """
        pdf_files = [
            f for f in os.listdir(self.data_dir)
            if f.lower().endswith(".pdf")
        ]

        if not pdf_files:
            logger.warning(f"No PDF files found in {self.data_dir}")
            return {}

        logger.info(f"Found {len(pdf_files)} PDF(s) to process.")
        results = {}

        for pdf_file in pdf_files:
            pdf_path = os.path.join(self.data_dir, pdf_file)
            try:
                raw   = self.extract_text(pdf_path)
                clean = self.clean_text(raw)
                path  = self.save_text(pdf_file, clean)
                results[pdf_file] = {
                    "raw_text"  : raw,
                    "clean_text": clean,
                    "txt_path"  : path,
                }
            except (FileNotFoundError, RuntimeError) as exc:
                logger.error(f"Skipping '{pdf_file}': {exc}")

        logger.info(f"Processing complete. {len(results)}/{len(pdf_files)} PDFs succeeded.")
        return results

    # ── Single-file convenience method ───────────────────────

    def process_single(self, pdf_path: str = None, stream: bytes = None, filename: str = None) -> dict:
        """
        Extract, clean, save and return result for one PDF.

        Args:
            pdf_path: Path to the PDF file (optional).
            stream: Raw bytes of the PDF file (optional).
            filename: Override the filename (optional).

        Returns:
            Dict with keys: raw_text, clean_text, txt_path.
        """
        if filename is None:
            filename = os.path.basename(pdf_path) if pdf_path else "document.pdf"
            
        raw_text  = self.extract_text(pdf_path=pdf_path, stream=stream)
        clean     = self.clean_text(raw_text)
        txt_path  = self.save_text(filename, clean)
        return {
            "raw_text"  : raw_text,
            "clean_text": clean,
            "txt_path"  : txt_path,
        }


# ── Quick test (run this file directly) ─────────────────────
if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1:
        pdf = sys.argv[1]
        processor = PDFProcessor()
        result = processor.process_single(pdf)
        print(f"\n✅ Extracted {len(result['clean_text']):,} characters.")
        print("First 500 chars:\n", result["clean_text"][:500])
    else:
        processor = PDFProcessor()
        results = processor.process_all()
        print(f"\n✅ Processed {len(results)} PDF(s).")


In [ ]:
%%writefile src/chunking.py
# ============================================================
# src/chunking.py
# Phase 2 — Text Chunking
#
# Objective:
#   Split long cleaned text into smaller, overlapping chunks
#   suitable for embedding and retrieval.
#
# Architecture:
#   TextChunker class
#     └── chunk_text(text, metadata) -> list[dict]
#     └── chunk_documents(docs_dict) -> list[dict]
#
# Why chunking matters:
#   Embedding models have a token limit (~512 tokens for
#   MiniLM). Splitting with overlap ensures context is not
#   lost at chunk boundaries.
#
# Dependencies: langchain, logging
# ============================================================

import logging
import os

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import CHUNK_SIZE, CHUNK_OVERLAP, LOG_LEVEL

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class TextChunker:
    """
    Splits research paper text into overlapping chunks.

    Each chunk is a dict:
      {
          "chunk_id"   : int,           # global unique index
          "source"     : str,           # source PDF filename
          "text"       : str,           # the chunk content
          "char_start" : int,           # approx start position
      }

    Usage:
        chunker = TextChunker()
        chunks = chunker.chunk_text(clean_text, source="paper.pdf")
    """

    def __init__(
        self,
        chunk_size: int = CHUNK_SIZE,
        chunk_overlap: int = CHUNK_OVERLAP
    ):
        self.chunk_size    = chunk_size
        self.chunk_overlap = chunk_overlap

        # RecursiveCharacterTextSplitter tries to split on
        # paragraph → sentence → word boundaries in order,
        # falling back to characters only as a last resort.
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=len,
        )
        logger.info(
            f"TextChunker ready | chunk_size={chunk_size} | overlap={chunk_overlap}"
        )

    # ── Single document ──────────────────────────────────────

    def chunk_text(self, text: str, source: str = "unknown") -> list:
        """
        Chunk a single document's text.

        Args:
            text   : Cleaned text string.
            source : Filename/label for provenance tracking.

        Returns:
            List of chunk dicts (see class docstring).
        """
        if not text or not text.strip():
            logger.warning(f"Empty text provided for source: {source}")
            return []

        raw_chunks = self.splitter.split_text(text)

        chunks = []
        for idx, chunk_text in enumerate(raw_chunks):
            chunks.append({
                "chunk_id"  : idx,
                "source"    : source,
                "text"      : chunk_text.strip(),
                "char_start": text.find(chunk_text[:50]) if len(chunk_text) >= 50 else 0,
            })

        logger.info(
            f"'{source}' → {len(chunks)} chunks "
            f"(avg {sum(len(c['text']) for c in chunks) // max(len(chunks),1)} chars each)"
        )
        return chunks

    # ── Multiple documents ───────────────────────────────────

    def chunk_documents(self, docs_dict: dict) -> list:
        """
        Chunk multiple documents and return a flat list with
        globally unique chunk IDs.

        Args:
            docs_dict: Output of PDFProcessor.process_all()
                       {filename: {"clean_text": str, ...}}

        Returns:
            Flat list of all chunks across all documents,
            with globally sequential chunk_ids.
        """
        all_chunks = []
        global_id  = 0

        for filename, doc_data in docs_dict.items():
            clean_text = doc_data.get("clean_text", "")
            chunks     = self.chunk_text(clean_text, source=filename)

            # Re-number chunk_ids globally
            for chunk in chunks:
                chunk["chunk_id"] = global_id
                global_id += 1

            all_chunks.extend(chunks)

        logger.info(f"Total chunks across all documents: {len(all_chunks)}")
        return all_chunks

    # ── Utility ──────────────────────────────────────────────

    def chunk_from_file(self, txt_path: str) -> list:
        """
        Load a saved .txt file and chunk it.

        Args:
            txt_path: Path to a plain-text file.

        Returns:
            List of chunk dicts.
        """
        if not os.path.isfile(txt_path):
            raise FileNotFoundError(f"Text file not found: {txt_path}")

        source = os.path.basename(txt_path)
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read()

        return self.chunk_text(text, source=source)


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    sample = """
    Attention Is All You Need

    Abstract
    The dominant sequence transduction models are based on complex recurrent or
    convolutional neural networks that include an encoder and a decoder.
    The best performing models also connect the encoder and decoder through an
    attention mechanism. We propose a new simple network architecture, the
    Transformer, based solely on attention mechanisms, dispensing with recurrence
    and convolutions entirely.

    1. Introduction
    Recurrent neural networks, long short-term memory and gated recurrent neural
    networks in particular, have been firmly established as state of the art
    approaches in sequence modelling and transduction problems such as language
    modelling and machine translation.
    """ * 10  # repeat to create enough text for chunking

    chunker = TextChunker(chunk_size=200, chunk_overlap=40)
    chunks  = chunker.chunk_text(sample, source="test_paper.pdf")

    print(f"\n✅ Created {len(chunks)} chunks.\n")
    for c in chunks[:3]:
        print(f"  Chunk {c['chunk_id']} | {c['source']} | {len(c['text'])} chars")
        print(f"  Preview: {c['text'][:80]}...\n")


In [ ]:
%%writefile src/embeddings.py
# ============================================================
# src/embeddings.py
# Phase 3 — Embedding Generation
#
# Objective:
#   Convert text chunks into dense vector representations
#   (embeddings) using a local Sentence Transformer model.
#
# Architecture:
#   EmbeddingGenerator class
#     ├── embed_chunks(chunks)       -> np.ndarray  (N, D)
#     ├── embed_query(query)         -> np.ndarray  (D,)
#     ├── save_embeddings(arr, path)
#     └── load_embeddings(path)      -> np.ndarray
#
# Model: all-MiniLM-L6-v2
#   - 384-dimensional embeddings
#   - Fast on CPU, excellent quality for semantic search
#   - Downloads automatically from HuggingFace on first use
#
# Dependencies: sentence-transformers, numpy, tqdm, logging
# ============================================================

import os
import logging

import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import EMBEDDING_MODEL, EMBEDDINGS_DIR, LOG_LEVEL

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class EmbeddingGenerator:
    """
    Generates sentence embeddings for text chunks and queries.

    Usage:
        gen = EmbeddingGenerator()
        embeddings = gen.embed_chunks(chunks)   # shape: (N, 384)
        query_vec  = gen.embed_query("What is attention?")
    """

    def __init__(self, model_name: str = EMBEDDING_MODEL):
        logger.info(f"Loading embedding model: {model_name}")
        logger.info("  (First run downloads ~90 MB from HuggingFace — once only)")
        self.model_name = model_name
        self._model     = None
        self.dim        = None

    def _load_model(self):
        """Lazy-initialise the embedding model on first use."""
        if self._model is None:
            self._model = SentenceTransformer(self.model_name)
            self.dim    = self._model.get_sentence_embedding_dimension()
            logger.info(f"  Model ready | embedding dim = {self.dim}")

    @property
    def model(self):
        """Property that auto-loads the model on first access."""
        self._load_model()
        return self._model

    # ── Embed chunks ─────────────────────────────────────────

    def embed_chunks(self, chunks: list, batch_size: int = 32) -> np.ndarray:
        """
        Generate embeddings for a list of chunk dicts.

        Args:
            chunks    : List of chunk dicts with key 'text'.
            batch_size: How many texts to encode at once.

        Returns:
            numpy array of shape (N, embedding_dim).

        Raises:
            ValueError: If chunks list is empty.
        """
        if not chunks:
            raise ValueError("chunks list is empty — nothing to embed.")

        texts = [c["text"] for c in chunks]
        logger.info(f"Embedding {len(texts)} chunks (batch_size={batch_size}) …")

        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,  # L2-normalise for cosine similarity
        )

        logger.info(f"Embeddings shape: {embeddings.shape}")
        return embeddings

    # ── Embed single query ───────────────────────────────────

    def embed_query(self, query: str) -> np.ndarray:
        """
        Embed a single query string.

        Args:
            query: The user question or search string.

        Returns:
            1-D numpy array of shape (embedding_dim,).
        """
        if not query or not query.strip():
            raise ValueError("Query string is empty.")

        vec = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0]

        logger.debug(f"Query embedded | shape: {vec.shape}")
        return vec

    # ── Persistence ──────────────────────────────────────────

    def save_embeddings(self, embeddings: np.ndarray, name: str = "embeddings") -> str:
        """
        Save embeddings to disk as a .npy file.

        Args:
            embeddings: numpy array to save.
            name      : Base filename (without extension).

        Returns:
            Path to the saved file.
        """
        os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
        path = os.path.join(EMBEDDINGS_DIR, f"{name}.npy")
        np.save(path, embeddings)
        logger.info(f"Embeddings saved → {path}  ({embeddings.shape})")
        return path

    def load_embeddings(self, name: str = "embeddings") -> np.ndarray:
        """
        Load embeddings from disk.

        Args:
            name: Base filename (without .npy extension).

        Returns:
            numpy array of embeddings.

        Raises:
            FileNotFoundError: If the .npy file does not exist.
        """
        path = os.path.join(EMBEDDINGS_DIR, f"{name}.npy")
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Embeddings file not found: {path}")
        embeddings = np.load(path)
        logger.info(f"Embeddings loaded ← {path}  ({embeddings.shape})")
        return embeddings


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    sample_chunks = [
        {"chunk_id": 0, "source": "test.pdf", "text": "Attention is all you need."},
        {"chunk_id": 1, "source": "test.pdf", "text": "Transformers changed NLP forever."},
        {"chunk_id": 2, "source": "test.pdf", "text": "BERT uses bidirectional attention."},
    ]

    gen = EmbeddingGenerator()
    emb = gen.embed_chunks(sample_chunks)
    print(f"\n✅ Chunk embeddings shape: {emb.shape}")

    qvec = gen.embed_query("What is the transformer model?")
    print(f"✅ Query embedding shape : {qvec.shape}")

    # Test cosine similarity (arrays are already normalised)
    scores = emb @ qvec
    for i, s in enumerate(scores):
        print(f"  Chunk {i} similarity: {s:.4f}")


In [ ]:
%%writefile src/vector_db.py
# ============================================================
# src/vector_db.py
# Phase 4 — FAISS Vector Store
#
# Objective:
#   Build a FAISS index from embeddings, persist it to disk,
#   reload it, and perform fast nearest-neighbour search.
#
# Architecture:
#   VectorDB class
#     ├── build_index(embeddings)
#     ├── save(index_path, meta_path)
#     ├── load(index_path, meta_path)
#     ├── search(query_vec, top_k) -> list[dict]
#     └── add_embeddings(new_embeddings, new_chunks)
#
# Why FAISS?
#   FAISS (Facebook AI Similarity Search) provides highly
#   optimised C++ similarity search accessible from Python.
#   IndexFlatIP uses inner-product (= cosine when vectors
#   are L2-normalised) — perfect for our use case.
#
# Dependencies: faiss-cpu, numpy, json, logging
# ============================================================

import os
import json
import logging

import numpy as np
import faiss

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import FAISS_INDEX_FILE, CHUNKS_META_FILE, TOP_K_RESULTS, LOG_LEVEL

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class VectorDB:
    """
    Wraps a FAISS index alongside chunk metadata for
    building, saving, loading, and querying a vector store.

    Usage:
        db = VectorDB()
        db.build_index(embeddings, chunks)
        db.save()
        # --- later ---
        db = VectorDB()
        db.load()
        results = db.search(query_vec, top_k=5)
    """

    def __init__(
        self,
        index_path: str = FAISS_INDEX_FILE,
        meta_path : str = CHUNKS_META_FILE,
    ):
        self.index_path = index_path
        self.meta_path  = meta_path
        self.index      = None   # faiss.Index instance
        self.chunks     = []     # list of chunk dicts (metadata)
        logger.info("VectorDB initialised.")

    # ── Build ────────────────────────────────────────────────

    def build_index(self, embeddings: np.ndarray, chunks: list) -> None:
        """
        Create a new FAISS index from embeddings.

        Args:
            embeddings: numpy array of shape (N, D), L2-normalised.
            chunks    : List of chunk dicts matching the embeddings.

        Raises:
            ValueError: If embeddings is empty or shapes mismatch.
        """
        if embeddings is None or len(embeddings) == 0:
            raise ValueError("Cannot build index from empty embeddings.")
        if len(embeddings) != len(chunks):
            raise ValueError(
                f"Mismatch: {len(embeddings)} embeddings vs {len(chunks)} chunks."
            )

        dim = embeddings.shape[1]
        logger.info(f"Building FAISS IndexFlatIP | dim={dim} | vectors={len(embeddings)}")

        # IndexFlatIP = exact inner-product search.
        # With L2-normalised vectors, IP == cosine similarity.
        self.index  = faiss.IndexFlatIP(dim)
        self.chunks = chunks

        # FAISS requires float32
        embeddings_f32 = embeddings.astype(np.float32)
        self.index.add(embeddings_f32)

        logger.info(f"Index built. Total vectors stored: {self.index.ntotal}")

    # ── Save ─────────────────────────────────────────────────

    def save(
        self,
        index_path: str | None = None,
        meta_path : str | None = None,
    ) -> None:
        """
        Persist the FAISS index and chunk metadata to disk.

        Args:
            index_path: Override default index file path.
            meta_path : Override default metadata file path.

        Raises:
            RuntimeError: If the index has not been built yet.
        """
        if self.index is None:
            raise RuntimeError("No index to save. Call build_index() first.")

        idx_path  = index_path or self.index_path
        meta_path = meta_path  or self.meta_path

        os.makedirs(os.path.dirname(idx_path),  exist_ok=True)
        os.makedirs(os.path.dirname(meta_path), exist_ok=True)

        faiss.write_index(self.index, idx_path)
        logger.info(f"FAISS index saved → {idx_path}")

        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(self.chunks, f, ensure_ascii=False, indent=2)
        logger.info(f"Chunk metadata saved → {meta_path}")

    # ── Load ─────────────────────────────────────────────────

    def load(
        self,
        index_path: str | None = None,
        meta_path : str | None = None,
    ) -> None:
        """
        Load FAISS index and chunk metadata from disk.

        Raises:
            FileNotFoundError: If either file does not exist.
        """
        idx_path  = index_path or self.index_path
        meta_path = meta_path  or self.meta_path

        if not os.path.isfile(idx_path):
            raise FileNotFoundError(f"FAISS index not found: {idx_path}")
        if not os.path.isfile(meta_path):
            raise FileNotFoundError(f"Chunks metadata not found: {meta_path}")

        self.index = faiss.read_index(idx_path)
        logger.info(f"FAISS index loaded ← {idx_path}  ({self.index.ntotal} vectors)")

        with open(meta_path, "r", encoding="utf-8") as f:
            self.chunks = json.load(f)
        logger.info(f"Chunk metadata loaded ← {meta_path}  ({len(self.chunks)} chunks)")

    # ── Search ───────────────────────────────────────────────

    def search(self, query_vec: np.ndarray, top_k: int = TOP_K_RESULTS) -> list:
        """
        Find the top-k most similar chunks to a query vector.

        Args:
            query_vec: 1-D numpy array (embedding_dim,), L2-normalised.
            top_k    : Number of results to return.

        Returns:
            List of result dicts:
            [
              {
                "rank"   : 1,
                "score"  : 0.92,        # cosine similarity
                "chunk_id": 42,
                "source" : "paper.pdf",
                "text"   : "...",
              },
              ...
            ]

        Raises:
            RuntimeError: If index is not loaded.
        """
        if self.index is None:
            raise RuntimeError("Index not loaded. Call build_index() or load() first.")

        query_f32 = query_vec.astype(np.float32).reshape(1, -1)
        scores, indices = self.index.search(query_f32, top_k)

        results = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
            if idx == -1:               # FAISS returns -1 for missing results
                continue
            chunk = self.chunks[idx].copy()
            chunk["rank"]  = rank
            chunk["score"] = float(score)
            results.append(chunk)

        logger.debug(f"Search returned {len(results)} results.")
        return results

    # ── Incremental add ──────────────────────────────────────

    def add_embeddings(self, new_embeddings: np.ndarray, new_chunks: list) -> None:
        """
        Add more embeddings to an existing index (incremental update).

        Args:
            new_embeddings: numpy array (M, D).
            new_chunks    : Corresponding chunk dicts (length M).
        """
        if self.index is None:
            raise RuntimeError("Index not initialised. Call build_index() first.")

        self.index.add(new_embeddings.astype(np.float32))
        self.chunks.extend(new_chunks)
        logger.info(
            f"Added {len(new_chunks)} chunks. Total: {self.index.ntotal} vectors."
        )

    # ── Utility ──────────────────────────────────────────────

    @property
    def is_ready(self) -> bool:
        """True if the index is built/loaded and non-empty."""
        return self.index is not None and self.index.ntotal > 0

    def get_stats(self) -> dict:
        """Return basic statistics about the current index."""
        return {
            "total_vectors": self.index.ntotal if self.index else 0,
            "total_chunks" : len(self.chunks),
            "sources"      : list({c.get("source", "?") for c in self.chunks}),
        }


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    import sys
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
    from src.embeddings import EmbeddingGenerator

    chunks = [
        {"chunk_id": 0, "source": "test.pdf", "text": "Attention is all you need."},
        {"chunk_id": 1, "source": "test.pdf", "text": "Transformers use multi-head attention."},
        {"chunk_id": 2, "source": "test.pdf", "text": "BERT fine-tunes on downstream tasks."},
    ]

    gen = EmbeddingGenerator()
    emb = gen.embed_chunks(chunks)

    db = VectorDB()
    db.build_index(emb, chunks)
    db.save()

    print("\n✅ Index built and saved.")
    print(f"   Stats: {db.get_stats()}")

    # Reload and search
    db2 = VectorDB()
    db2.load()
    qvec = gen.embed_query("What model uses self-attention?")
    results = db2.search(qvec, top_k=2)

    print("\n🔍 Search results:")
    for r in results:
        print(f"  Rank {r['rank']} | Score {r['score']:.4f} | {r['source']}")
        print(f"  Text: {r['text']}\n")


In [ ]:
%%writefile src/retriever.py
# ============================================================
# src/retriever.py
# Phase 5 — Semantic Search / Retriever
#
# Objective:
#   Provide a clean, high-level API that takes a natural-
#   language question and returns the most relevant text
#   chunks from the FAISS index.
#
# Architecture:
#   Retriever class
#     └── retrieve(query, top_k) -> list[dict]
#     └── retrieve_with_context(query, top_k) -> str (joined)
#
# This module is the "R" in RAG — it bridges the query
# and the vector store, formatting results for the LLM.
#
# Dependencies: embeddings.py, vector_db.py
# ============================================================

import os
import logging

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import TOP_K_RESULTS, LOG_LEVEL
from src.embeddings import EmbeddingGenerator
from src.vector_db  import VectorDB

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class Retriever:
    """
    Semantic search over the FAISS index.

    Usage:
        retriever = Retriever()
        retriever.load_index()
        results = retriever.retrieve("What are the key findings?")
        context = retriever.retrieve_with_context("What is BERT?")
    """

    def __init__(
        self,
        embedding_gen: EmbeddingGenerator | None = None,
        vector_db    : VectorDB | None = None,
    ):
        """
        Args:
            embedding_gen: Pre-initialised EmbeddingGenerator.
                           If None, a new one is created automatically.
            vector_db    : Pre-initialised VectorDB.
                           If None, a new one is created; call load_index().
        """
        self.embedding_gen = embedding_gen or EmbeddingGenerator()
        self.vector_db     = vector_db     or VectorDB()
        logger.info("Retriever initialised.")

    # ── Index management ─────────────────────────────────────

    def load_index(self) -> None:
        """Load the FAISS index and chunk metadata from disk."""
        self.vector_db.load()
        logger.info("Retriever: index loaded and ready.")

    def is_ready(self) -> bool:
        """Return True if the index is loaded and non-empty."""
        return self.vector_db.is_ready

    # ── Core retrieval ───────────────────────────────────────

    def retrieve(self, query: str, top_k: int = TOP_K_RESULTS) -> list:
        """
        Embed a query and return the top-k matching chunks.

        Args:
            query  : Natural language question or search string.
            top_k  : Number of results to return.

        Returns:
            List of chunk dicts, each with a 'score' field.

        Raises:
            RuntimeError: If the index is not ready.
            ValueError  : If query is empty.
        """
        if not query or not query.strip():
            raise ValueError("Query must not be empty.")
        if not self.is_ready():
            raise RuntimeError(
                "Vector index is not ready. Call load_index() or build the index first."
            )

        logger.info(f"Retrieving top-{top_k} chunks for query: '{query[:60]}...' ")
        query_vec = self.embedding_gen.embed_query(query)
        results   = self.vector_db.search(query_vec, top_k=top_k)

        logger.info(f"  Retrieved {len(results)} chunks.")
        for r in results:
            logger.debug(
                f"  Rank {r['rank']} | score={r['score']:.4f} | source={r['source']}"
            )

        return results

    # ── Context string for LLM prompt ────────────────────────

    def retrieve_with_context(
        self,
        query : str,
        top_k : int = TOP_K_RESULTS,
        sep   : str = "\n\n---\n\n",
    ) -> str:
        """
        Retrieve top-k chunks and join them into a single
        context string for use in an LLM prompt.

        Args:
            query: The user's question.
            top_k: Number of chunks to include.
            sep  : Separator between chunks.

        Returns:
            A multi-paragraph string with source annotations.
        """
        results  = self.retrieve(query, top_k=top_k)
        parts    = []

        for r in results:
            header = f"[Source: {r['source']} | Rank: {r['rank']} | Score: {r['score']:.3f}]"
            parts.append(f"{header}\n{r['text']}")

        context = sep.join(parts)
        logger.debug(f"Context length: {len(context):,} characters")
        return context

    # ── Source-filtered retrieval ─────────────────────────────

    def retrieve_from_source(
        self,
        query : str,
        source: str,
        top_k : int = TOP_K_RESULTS,
    ) -> list:
        """
        Retrieve chunks restricted to a specific source document.

        Args:
            query : The search query.
            source: Filename of the paper to restrict to.
            top_k : Number of results to return.

        Returns:
            Filtered list of chunk dicts.
        """
        all_results = self.retrieve(query, top_k=top_k * 3)  # over-fetch then filter
        filtered = [r for r in all_results if r.get("source") == source]
        return filtered[:top_k]


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    # This test assumes vector_db.py was run first to build the index
    retriever = Retriever()

    if not retriever.is_ready():
        print("⚠️  Index not found — run vector_db.py first to build the index.")
    else:
        results = retriever.retrieve("What is the attention mechanism?", top_k=3)
        print(f"\n✅ Retrieved {len(results)} results.\n")
        for r in results:
            print(f"  Rank {r['rank']} | Score {r['score']:.4f} | {r['source']}")
            print(f"  {r['text'][:120]}…\n")


In [ ]:
%%writefile src/rag_pipeline.py
# ============================================================
# src/rag_pipeline.py
# Phase 6 — Retrieval-Augmented Generation (RAG) Pipeline
#
# Objective:
#   Answer user questions by:
#     1. Retrieving relevant context chunks (Retriever)
#     2. Building a prompt with context + question
#     3. Running a local HuggingFace QA model for the answer
#
# Architecture:
#   RAGPipeline class
#     ├── answer(question, top_k) -> dict
#     │     returns: answer, context, sources, confidence
#     └── _build_prompt(question, context) -> str
#
# Model: deepset/roberta-base-squad2
#   - Extractive QA: finds the answer span within the context
#   - Works fully offline after first download
#   - Fast on CPU
#
# For generative answers (replacing extractive QA with a
# seq2seq model), swap to google/flan-t5-base and use the
# text2text-generation pipeline.
#
# Dependencies: transformers, retriever.py
# ============================================================

import os
import logging

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import QA_MODEL, TOP_K_RESULTS, LOG_LEVEL
from src.retriever import Retriever

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TextIteratorStreamer
from threading import Thread

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class RAGPipeline:
    """
    End-to-end Retrieval-Augmented Generation pipeline.

    Combine semantic retrieval with extractive QA to answer
    natural language questions from your research papers.

    Usage:
        rag = RAGPipeline()
        result = rag.answer("What dataset was used for evaluation?")
        print(result["answer"])
        print(result["sources"])
    """

    def __init__(
        self,
        retriever : Retriever | None = None,
        qa_model  : str = QA_MODEL,
    ):
        """
        Args:
            retriever: Pre-initialised Retriever.
                       If None, a new one is created; index auto-loaded.
            qa_model : HuggingFace model ID for QA.
        """
        # ── Retriever setup ──
        if retriever is not None:
            self.retriever = retriever
        else:
            self.retriever = Retriever()
            # Try to load the index; warn if not available yet
            try:
                self.retriever.load_index()
            except FileNotFoundError:
                logger.warning(
                    "No vector index found. "
                    "Call build_index_from_pdfs() or ingest_pdfs() first."
                )

        # ── QA model setup (lazy) ──
        self.qa_model_name = qa_model
        self.tokenizer = None
        self.model = None
        logger.info(f"RAGPipeline initialised (QA model: {qa_model}, loaded on first use).")

    def _load_model(self):
        """Lazy-initialise the QA model on first use."""
        if self.model is None:
            logger.info(f"Loading QA model: {self.qa_model_name}")
            self.tokenizer = AutoTokenizer.from_pretrained(self.qa_model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.qa_model_name)
            logger.info("QA model ready.")

    # ── Core QA ──────────────────────────────────────────────

    def answer(self, question: str, top_k: int = TOP_K_RESULTS, stream: bool = False) -> dict:
        """
        Answer a question using retrieved context.

        Args:
            question: The user's natural language question.
            top_k   : Number of context chunks to retrieve.
            stream  : If True, returns a 'streamer' and 'thread' instead of 'answer'.

        Returns:
            {
                "question"  : str,
                "answer"    : str,   # if stream=False
                "streamer"  : TextIteratorStreamer, # if stream=True
                "thread"    : Thread, # if stream=True
                "confidence": float,
                "context"   : str,
                "sources"   : list,
            }

        Raises:
            RuntimeError: If the retriever index is not ready.
            ValueError  : If the question is empty.
        """
        if not question or not question.strip():
            raise ValueError("Question must not be empty.")

        if not self.retriever.is_ready():
            raise RuntimeError(
                "No vector index found. Please upload and index papers first."
            )

        # Step 0: Lazy-load the QA model if needed
        self._load_model()

        # Step 1: Retrieve relevant chunks
        logger.info(f"RAG → Question: '{question[:80]}'")
        chunks = self.retriever.retrieve(question, top_k=top_k)

        # Step 2: Build context from retrieved chunks
        context_parts = []
        for i, c in enumerate(chunks, 1):
            context_parts.append(f"[Source {i}]: {c['text']}")
        context = "\n\n".join(context_parts)
        sources = [{"source": c["source"], "score": c["score"]} for c in chunks]

        # Step 3: Run Generative QA model
        logger.info(f"Running QA model on context ({len(context)} chars) …")
        prompt = f"Answer the following question based ONLY on the provided context. If the context does not contain the answer, explicitly state 'I cannot answer this based on the provided text.' Do not use outside knowledge.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)

        if stream:
            streamer = TextIteratorStreamer(self.tokenizer, skip_prompt=True, skip_special_tokens=True)
            generation_kwargs = dict(
                **inputs,
                streamer=streamer,
                max_new_tokens=250,
                do_sample=True,
                temperature=0.3,
            )
            thread = Thread(target=self.model.generate, kwargs=generation_kwargs)
            thread.start()

            return {
                "question"  : question,
                "streamer"  : streamer,
                "thread"    : thread,
                "confidence": 1.0,
                "context"   : context,
                "sources"   : sources,
            }
        else:
            try:
                with torch.no_grad():
                    outputs = self.model.generate(
                        **inputs,
                        max_new_tokens=250,
                        num_beams=4,
                        early_stopping=True
                    )
                answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                confidence = 1.0
            except Exception as exc:
                logger.error(f"QA model failed: {exc}")
                answer     = "Could not generate an answer. Please rephrase your question."
                confidence = 0.0

            logger.info(f"Answer: '{answer[:80]}'")

            return {
                "question"  : question,
                "answer"    : answer,
                "confidence": confidence,
                "context"   : context,
                "sources"   : sources,
            }

    # ── Full ingest pipeline (convenience) ───────────────────

    def ingest_and_build_index(
        self,
        pdf_paths: list,
        force_rebuild: bool = False,
    ) -> dict:
        """
        Full pipeline: extract PDFs → chunk → embed → build FAISS index.
        Call this once when new PDFs are uploaded.

        Args:
            pdf_paths    : List of paths to PDF files.
            force_rebuild: If True, rebuild even if index exists.

        Returns:
            {
                "num_pdfs"  : int,
                "num_chunks": int,
                "index_ready": bool,
            }
        """
        import shutil
        from src.pdf_processor import PDFProcessor
        from src.chunking      import TextChunker
        from src.embeddings    import EmbeddingGenerator
        from src.vector_db     import VectorDB
        from config            import DATA_DIR

        logger.info(f"Ingesting {len(pdf_paths)} PDF(s) …")

        # Copy PDFs to data dir
        os.makedirs(DATA_DIR, exist_ok=True)
        for p in pdf_paths:
            dest = os.path.join(DATA_DIR, os.path.basename(p))
            if os.path.abspath(p) != os.path.abspath(dest):
                if not os.path.isfile(dest) or force_rebuild:
                    shutil.copy2(p, dest)

        # Phase 1 — Extract
        processor = PDFProcessor()
        docs      = {}
        for p in pdf_paths:
            filename = os.path.basename(p)
            result   = processor.process_single(p)
            docs[filename] = result

        # Phase 2 — Chunk
        chunker    = TextChunker()
        all_chunks = chunker.chunk_documents(docs)

        # Phase 3 — Embed
        gen        = EmbeddingGenerator()
        embeddings = gen.embed_chunks(all_chunks)
        gen.save_embeddings(embeddings)

        # Phase 4 — Index
        db = VectorDB()
        db.build_index(embeddings, all_chunks)
        db.save()

        # Update retriever's reference
        self.retriever.vector_db = db
        logger.info("Ingest complete. Index ready.")

        return {
            "num_pdfs"   : len(pdf_paths),
            "num_chunks" : len(all_chunks),
            "index_ready": True,
        }


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    rag = RAGPipeline()
    if rag.retriever.is_ready():
        result = rag.answer("What is the main contribution of this paper?")
        print(f"\n✅ Answer     : {result['answer']}")
        print(f"   Confidence : {result['confidence']:.3f}")
        print(f"   Sources    : {[s['source'] for s in result['sources']]}")
    else:
        print("⚠️  No index found. Please ingest PDFs first.")


In [ ]:
%%writefile src/summarizer.py
# ============================================================
# src/summarizer.py
# Phase 7 & 8 — Summarization + Research Insights Extraction
#
# Objective:
#   1. Generate a concise abstract-style summary of a paper.
#   2. Extract structured research insights:
#      - Key Findings
#      - Limitations
#      - Future Work
#
# Architecture:
#   Summarizer class
#     ├── summarize(text)               -> str
#     ├── extract_insights(text)        -> dict
#     └── full_analysis(chunks, source) -> dict
#
# Strategy:
#   - Use BART (facebook/bart-large-cnn) for summarization.
#   - Use keyword-pattern extraction as a reliable, fast way
#     to pull structured sections without needing a GPU.
#   - Fallback: retrieval-based extraction when sections
#     are not explicitly labelled.
#
# Dependencies: transformers, retriever.py (optional)
# ============================================================

import re
import os
import logging

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import SUMMARIZATION_MODEL, LOG_LEVEL

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)

# ── Section header patterns (case-insensitive) ────────────────
_FINDINGS_PATTERNS  = [
    r"(?i)(key\s+(?:findings?|contributions?|results?|insights?|discoveries))",
    r"(?i)(main\s+(?:findings?|contributions?|results?))",
    r"(?i)(conclusion[s]?|summary\s+of\s+(?:findings?|results?))",
    r"(?i)(our\s+(?:results?|findings?|contributions?))",
]

_LIMITATIONS_PATTERNS = [
    r"(?i)(limitation[s]?)",
    r"(?i)(drawback[s]?|weakness(?:es)?)",
    r"(?i)(constraint[s]?)",
    r"(?i)(we\s+do\s+not\s+address|beyond\s+the\s+scope)",
]

_FUTURE_WORK_PATTERNS = [
    r"(?i)(future\s+work|future\s+direction[s]?|future\s+research)",
    r"(?i)(open\s+problem[s]?|open\s+question[s]?)",
    r"(?i)(further\s+research|further\s+investigation)",
    r"(?i)(promising\s+(?:avenue[s]?|direction[s]?))",
]


class Summarizer:
    """
    Summarizes research papers and extracts structured insights.

    Usage:
        summ = Summarizer()
        summary   = summ.summarize(full_text)
        insights  = summ.extract_insights(full_text)
        analysis  = summ.full_analysis(chunks, source="paper.pdf")
    """

    def __init__(self, model_name: str = SUMMARIZATION_MODEL):
        logger.info(f"Loading summarization model: {model_name}")
        logger.info("  (First run downloads ~1.6 GB from HuggingFace — once only)")
        self.model_name = model_name
        self.tokenizer = None
        self.model = None

    def _load_model(self):
        """Lazy-initialise the summarization model."""
        if self.model is None:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
            logger.info("Summarization model loaded.")

    # ── Summarization ────────────────────────────────────────

    def summarize(
        self,
        text        : str,
        max_length  : int = 300,
        min_length  : int = 80,
        chunk_limit : int = 3000,   # chars to feed to BART (token limit ~1024)
    ) -> str:
        """
        Generate a concise summary of the provided text.

        BART has a token limit, so we truncate to chunk_limit
        characters before passing to the model.

        Args:
            text       : Full paper or section text.
            max_length : Max summary tokens.
            min_length : Min summary tokens.
            chunk_limit: Max input characters (BART token budget).

        Returns:
            A concise summary string.
        """
        if not text or not text.strip():
            return "No text provided for summarization."

        # Truncate to stay within model token limit
        input_text = text[:chunk_limit]

        logger.info(f"Summarizing {len(input_text):,} characters …")
        try:
            self._load_model()
            inputs = self.tokenizer([input_text], return_tensors="pt", max_length=1024, truncation=True)
            summary_ids = self.model.generate(
                inputs["input_ids"],
                max_length=max_length,
                min_length=min_length,
                early_stopping=True
            )
            summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            logger.info(f"Summary generated ({len(summary)} chars).")
            return summary
        except Exception as exc:
            logger.error(f"Summarization failed: {exc}")
            return f"Summarization error: {exc}"

    # ── Section extraction ───────────────────────────────────

    def _extract_section(self, text: str, patterns: list) -> list:
        """
        Extract sentences that follow a matched section header.

        Strategy:
          1. Find the first heading that matches any pattern.
          2. Extract up to 1000 characters.
          3. Split into sentences and discard any incomplete trailing sentence.

        Args:
            text    : Full paper text.
            patterns: List of regex patterns for section headers.

        Returns:
            List of sentences.
        """
        for pattern in patterns:
            match = re.search(pattern, text)
            if match:
                start = match.end() # Start AFTER the heading
                snippet = text[start : start + 1000]
                # Clean up newlines
                snippet = snippet.replace('\n', ' ')
                # Split into sentences
                sentences = re.split(r'(?<=[.!?])\s+', snippet)
                
                # Discard the last sentence if it's incomplete
                if sentences and not re.search(r'[.!?]$', sentences[-1].strip()):
                    sentences = sentences[:-1]
                
                # Filter out very short fragments
                cleaned = [s.strip() for s in sentences if len(s.strip()) > 20]
                if cleaned:
                    return cleaned

        return []

    def _extract_bullet_sentences(self, text: str, keywords: list) -> list:
        """
        Find sentences containing specific keywords and return as list.

        Fallback extraction when sections aren't explicitly labelled.
        """
        sentences = re.split(r'(?<=[.!?])\s+', text)
        hits = []
        for sent in sentences:
            sent_lower = sent.lower()
            if any(kw.lower() in sent_lower for kw in keywords):
                cleaned = sent.strip()
                if len(cleaned) > 20:  # skip very short fragments
                    hits.append(cleaned)
        return hits[:5]  # return top 5 matches

    # ── Insights extraction ──────────────────────────────────

    def extract_insights(self, text: str) -> dict:
        """
        Extract structured research insights from paper text.

        Returns:
            {
                "key_findings": str or list,
                "limitations" : str or list,
                "future_work" : str or list,
            }

        Strategy:
          - First, try to find explicit section headings.
          - If not found, fall back to sentence-level keyword search.
        """
        logger.info("Extracting research insights …")

        # ── Key Findings ──
        findings_text = self._extract_section(text, _FINDINGS_PATTERNS)
        if findings_text:
            key_findings = findings_text
        else:
            sentences = self._extract_bullet_sentences(
                text,
                ["we show", "we propose", "we demonstrate", "our method",
                 "achieves", "outperforms", "significantly", "novel", "key finding"]
            )
            key_findings = sentences if sentences else ["Not explicitly stated in the paper."]

        # ── Limitations ──
        limit_text = self._extract_section(text, _LIMITATIONS_PATTERNS)
        if limit_text:
            limitations = limit_text
        else:
            sentences = self._extract_bullet_sentences(
                text,
                ["limitation", "drawback", "weakness", "constraint",
                 "does not", "cannot", "unable to", "restricted to"]
            )
            limitations = sentences if sentences else ["Not explicitly stated in the paper."]

        # ── Future Work ──
        future_text = self._extract_section(text, _FUTURE_WORK_PATTERNS)
        if future_text:
            future_work = future_text
        else:
            sentences = self._extract_bullet_sentences(
                text,
                ["future work", "future research", "further investigation",
                 "open problem", "promising direction", "plan to", "will explore"]
            )
            future_work = sentences if sentences else ["Not explicitly stated in the paper."]

        logger.info("Insights extraction complete.")
        return {
            "key_findings": key_findings,
            "limitations" : limitations,
            "future_work" : future_work,
        }

    # ── Full analysis (convenience) ──────────────────────────

    def full_analysis(self, chunks: list, source: str = "paper") -> dict:
        """
        Run summary + insights on a set of chunks from one paper.

        Args:
            chunks: List of chunk dicts with 'text' and 'source'.
            source: Name of the paper for logging.

        Returns:
            {
                "source"      : str,
                "summary"     : str,
                "key_findings": str or list,
                "limitations" : str or list,
                "future_work" : str or list,
            }
        """
        logger.info(f"Full analysis for: {source}")

        # Reconstruct full text from chunks
        full_text = "\n\n".join(
            c["text"] for c in chunks
            if c.get("source") == source or source == "all"
        )

        if not full_text.strip():
            full_text = "\n\n".join(c["text"] for c in chunks)

        summary  = self.summarize(full_text)
        insights = self.extract_insights(full_text)

        return {
            "source"      : source,
            "summary"     : summary,
            **insights,
        }


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    sample_text = """
    Abstract
    We propose the Transformer, a new model architecture based entirely on attention
    mechanisms, dispensing with recurrence and convolutions. The model achieves state-
    of-the-art results on machine translation tasks.

    Key Findings
    Our model achieves 28.4 BLEU on WMT 2014 English-to-German translation, improving
    over existing best results by over 2 BLEU. We demonstrate that Transformers
    generalize well to other tasks by applying them to English constituency parsing.

    Limitations
    The model has a limitation in handling very long sequences due to the quadratic
    memory complexity of self-attention. We do not address streaming or online inference.

    Future Work
    Future work will explore sparse attention mechanisms to reduce computational cost.
    We plan to investigate applying Transformers to video and audio tasks.
    """

    summ = Summarizer()
    print("\n📝 Summary:")
    print(summ.summarize(sample_text))

    print("\n🔍 Insights:")
    insights = summ.extract_insights(sample_text)
    for key, val in insights.items():
        print(f"\n  {key.upper()}:")
        if isinstance(val, list):
            for item in val:
                print(f"    • {item}")
        else:
            print(f"    {val}")


In [ ]:
%%writefile src/arxiv_search.py
# ============================================================
# src/arxiv_search.py
# Agentic Workflow: Scholarly Paper Discovery (arXiv API)
# Phase 2 — Robust arXiv Search & Metadata Extractor
# ============================================================

import os
import re
import logging
import urllib.request
import urllib.error
import arxiv
from typing import List, Dict, Any, Optional

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import DATA_DIR, ARXIV_SEARCH_LIMIT, LOG_LEVEL

logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class ArxivSearcher:
    """
    Independent paper discovery module using the official arXiv API.
    Retrieves scholarly paper metadata and manages PDF downloads.
    """
    def __init__(self, limit: int = ARXIV_SEARCH_LIMIT):
        self.limit = limit
        self.client = arxiv.Client(
            page_size=100,
            delay_seconds=3,
            num_retries=3
        )

    def clean_text(self, text: str) -> str:
        """Removes extra line breaks and collapses whitespace."""
        if not text:
            return ""
        cleaned = re.sub(r"\s+", " ", text.replace("\n", " "))
        return cleaned.strip()

    def search_papers(self, query: str, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        """
        Searches arXiv for papers matching the research query.
        Returns structured metadata list without downloading PDFs.
        
        Output format per paper:
        - title (str)
        - authors (List[str])
        - abstract (str)
        - publication_date (str YYYY-MM-DD)
        - year (int)
        - arxiv_id / id (str)
        - url (str PDF link)
        """
        max_results = limit if limit is not None else self.limit
        cleaned_query = self.clean_text(query.strip('\'"'))
        
        if not cleaned_query:
            logger.warning("Empty search query provided to ArxivSearcher.")
            return []

        logger.info(f"Querying arXiv API for: '{cleaned_query}' (limit: {max_results})")
        
        try:
            search = arxiv.Search(
                query=cleaned_query,
                max_results=max_results,
                sort_by=arxiv.SortCriterion.Relevance
            )
            
            results = []
            for result in self.client.results(search):
                arxiv_id = result.entry_id.split("/")[-1]
                pub_date = result.published.strftime("%Y-%m-%d") if result.published else "N/A"
                pub_year = result.published.year if result.published else 0
                
                pdf_url = result.pdf_url
                if pdf_url and not pdf_url.endswith(".pdf"):
                    pdf_url += ".pdf"

                paper_info = {
                    "title": self.clean_text(result.title),
                    "authors": [a.name for a in result.authors],
                    "abstract": self.clean_text(result.summary),
                    "publication_date": pub_date,
                    "year": pub_year,
                    "arxiv_id": arxiv_id,
                    "id": arxiv_id,
                    "url": pdf_url,
                    "local_path": os.path.join(DATA_DIR, f"{arxiv_id}.pdf")
                }
                results.append(paper_info)

            logger.info(f"Successfully discovered {len(results)} papers from arXiv.")
            return results

        except (urllib.error.URLError, TimeoutError) as net_err:
            logger.error(f"Network error during arXiv search for '{cleaned_query}': {net_err}")
            return []
        except Exception as exc:
            logger.error(f"Unexpected error querying arXiv API for '{cleaned_query}': {exc}")
            return []

    def download_paper_pdf(self, paper_info: Dict[str, Any]) -> Optional[str]:
        """
        Downloads PDF for a specific paper into data/ directory if not present.
        Returns the local file path on success, or None on failure.
        """
        os.makedirs(DATA_DIR, exist_ok=True)
        arxiv_id = paper_info.get("arxiv_id") or paper_info.get("id") or "paper"
        pdf_path = paper_info.get("local_path") or os.path.join(DATA_DIR, f"{arxiv_id}.pdf")
        
        if os.path.exists(pdf_path) and os.path.getsize(pdf_path) > 0:
            logger.info(f"Paper PDF already cached locally: {os.path.basename(pdf_path)}")
            return pdf_path

        download_url = paper_info.get("url")
        if not download_url:
            logger.warning(f"No PDF URL found for paper: {paper_info.get('title')}")
            return None

        try:
            logger.info(f"Downloading PDF for '{paper_info.get('title', '')[:40]}...' from {download_url}")
            req = urllib.request.Request(
                download_url,
                headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
            )
            with urllib.request.urlopen(req, timeout=30) as response, open(pdf_path, "wb") as out_file:
                out_file.write(response.read())
            logger.info(f"Downloaded PDF → {pdf_path}")
            return pdf_path
        except Exception as exc:
            logger.error(f"Failed to download PDF for '{paper_info.get('title')}': {exc}")
            if os.path.exists(pdf_path):
                os.remove(pdf_path)
            return None

    def search_and_download(self, query: str, limit: Optional[int] = None) -> List[Dict[str, Any]]:
        """
        Backwards-compatible method: Searches arXiv and downloads PDFs for discovered papers.
        """
        papers = self.search_papers(query, limit=limit)
        valid_papers = []
        for paper in papers:
            pdf_path = self.download_paper_pdf(paper)
            if pdf_path:
                paper["local_path"] = pdf_path
                valid_papers.append(paper)
        return valid_papers


In [ ]:
%%writefile src/ranker.py
# ============================================================
# src/ranker.py
# Agentic Workflow: Semantic Paper Ranking Node
# Phase 3 — Cosine Similarity Paper Ranker
# ============================================================

import os
import logging
import numpy as np
from typing import List, Dict, Any, Optional

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import AGENT_TOP_K, LOG_LEVEL
from src.embeddings import EmbeddingGenerator

logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class SemanticRanker:
    """
    Ranks discovered paper metadata against the research topic
    using SentenceTransformers ('all-MiniLM-L6-v2') cosine similarity.
    """
    def __init__(self, top_k: int = AGENT_TOP_K, embed_gen: Optional[EmbeddingGenerator] = None):
        self.top_k = top_k
        self.embed_gen = embed_gen or EmbeddingGenerator()

    def calculate_cosine_similarity(self, vec1: np.ndarray, vec2: np.ndarray) -> float:
        """Computes normalized cosine similarity between two 1D vectors."""
        norm1 = np.linalg.norm(vec1)
        norm2 = np.linalg.norm(vec2)
        if norm1 == 0 or norm2 == 0:
            return 0.0
        score = np.dot(vec1, vec2) / (norm1 * norm2)
        return float(np.clip(score, 0.0, 1.0))

    def rank_papers(
        self,
        topic: str,
        papers: List[Dict[str, Any]],
        top_k: Optional[int] = None
    ) -> List[Dict[str, Any]]:
        """
        Ranks papers by abstract similarity against the research topic.
        
        Args:
            topic: Research topic query string.
            papers: List of paper metadata dictionaries from arXiv discovery.
            top_k: Optional override for the number of top papers to select.

        Returns:
            Sorted list of papers containing assigned rank, similarity score,
            title, year, and URL.
        """
        if not papers:
            logger.warning("No papers provided for semantic ranking.")
            return []

        k = top_k if top_k is not None else self.top_k
        logger.info(f"Ranking {len(papers)} candidate papers against topic: '{topic}' (Top-{k})")

        try:
            # 1. Encode Research Topic
            topic_vec = self.embed_gen.model.encode(topic, convert_to_numpy=True)

            # 2. Encode Abstracts
            abstracts = [p.get("abstract", "") or p.get("title", "") for p in papers]
            abstract_vecs = self.embed_gen.model.encode(abstracts, convert_to_numpy=True)

            # 3. Calculate Cosine Similarity per paper
            for i, paper in enumerate(papers):
                score = self.calculate_cosine_similarity(topic_vec, abstract_vecs[i])
                paper["relevance_score"] = round(score, 4)

            # 4. Sort in descending order of relevance
            sorted_papers = sorted(papers, key=lambda x: x.get("relevance_score", 0.0), reverse=True)

            # 5. Assign 1-indexed rank and truncate to Top-K
            top_papers = []
            for rank_idx, paper in enumerate(sorted_papers[:k], start=1):
                paper_copy = dict(paper)
                paper_copy["rank"] = rank_idx
                top_papers.append(paper_copy)

            logger.info(f"Successfully ranked and selected Top-{len(top_papers)} papers.")
            return top_papers

        except Exception as exc:
            logger.error(f"Error during semantic paper ranking: {exc}")
            # Fallback: assign rank 1..N with 0 score
            fallback = []
            for rank_idx, p in enumerate(papers[:k], start=1):
                p_copy = dict(p)
                p_copy["rank"] = rank_idx
                p_copy["relevance_score"] = 0.0
                fallback.append(p_copy)
            return fallback


In [ ]:
%%writefile src/agent.py
# ============================================================
# src/agent.py
# Agentic AI Research Assistant — LangGraph Orchestrator
# Phase 1: Research Planner & State Graph Definition
# ============================================================

import os
import logging
from typing import TypedDict, List, Dict, Any

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import LOG_LEVEL
from langgraph.graph import StateGraph, END

from src.arxiv_search import ArxivSearcher
from src.ranker import SemanticRanker
from src.rag_pipeline import RAGPipeline
from src.summarizer import Summarizer
from src.analyzer import AgentAnalyzer

logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


# ── State Definition ─────────────────────────────────────────

class ResearchState(TypedDict):
    research_topic: str
    topic: str                      # Alias for backwards compatibility
    research_questions: List[str]   # Decomposed questions guiding the analysis
    discovered_papers: List[dict]   # Raw papers returned by arXiv search
    raw_papers: List[dict]          # Alias for backwards compatibility
    ranked_papers: List[dict]       # Papers sorted by semantic relevance
    selected_papers: List[dict]     # Top-K papers selected for RAG ingestion
    paper_analysis: List[dict]      # Structured insight dictionaries per paper
    analyses: List[dict]            # Alias for backwards compatibility
    comparison: str                 # Markdown / HTML comparison matrix
    literature_review: str          # Multi-paper structured synthesis
    status: str                     # Current execution status for UI
    errors: List[str]               # Accumulated non-fatal execution errors


# ── Nodes ────────────────────────────────────────────────────

def plan_research_node(state: ResearchState) -> ResearchState:
    """
    Stage 1: Research Planner Node
    Decomposes the research topic into structured guiding research questions.
    """
    topic = state.get("research_topic") or state.get("topic", "")
    logger.info(f"Node [1/6]: Research Planner for topic: '{topic}'")
    state["status"] = "Planning research scope and formulating key questions..."
    
    # Deterministic decomposition into core analytical dimensions
    questions = [
        f"What are the foundational methodologies and models proposed for {topic}?",
        f"What empirical datasets and evaluation metrics are used to validate {topic}?",
        f"What are the major performance outcomes, key findings, and limitations in current {topic} research?"
    ]
    
    state["research_questions"] = questions
    state["topic"] = topic
    state["research_topic"] = topic
    logger.info(f"Formulated {len(questions)} research questions.")
    return state


def search_papers_node(state: ResearchState) -> ResearchState:
    """
    Stage 2: Paper Search Node
    Queries arXiv API for papers matching the research topic.
    """
    topic = state.get("research_topic") or state.get("topic", "")
    logger.info(f"Node [2/6]: Paper Search for: '{topic}'")
    state["status"] = "Searching arXiv for relevant scholarly papers..."
    
    try:
        searcher = ArxivSearcher()
        papers = searcher.search_and_download(topic)
        state["discovered_papers"] = papers
        state["raw_papers"] = papers
        logger.info(f"Discovered {len(papers)} papers from arXiv.")
    except Exception as exc:
        err_msg = f"Paper Search Error: {str(exc)}"
        logger.error(err_msg)
        state["errors"].append(err_msg)
        state["discovered_papers"] = []
        state["raw_papers"] = []
        
    return state


def rank_papers_node(state: ResearchState) -> ResearchState:
    """
    Stage 3: Paper Ranking Node
    Ranks discovered papers using Sentence Transformers cosine similarity.
    """
    topic = state.get("research_topic") or state.get("topic", "")
    papers = state.get("discovered_papers") or state.get("raw_papers", [])
    logger.info(f"Node [3/6]: Paper Ranking for {len(papers)} papers against '{topic}'")
    state["status"] = "Semantically ranking papers against research topic..."
    
    if not papers:
        logger.warning("No papers to rank.")
        state["ranked_papers"] = []
        state["selected_papers"] = []
        return state

    try:
        ranker = SemanticRanker()
        ranked = ranker.rank_papers(topic, papers)
        state["ranked_papers"] = ranked
        state["selected_papers"] = ranked
        logger.info(f"Successfully ranked and selected top {len(ranked)} papers.")
    except Exception as exc:
        err_msg = f"Paper Ranking Error: {str(exc)}"
        logger.error(err_msg)
        state["errors"].append(err_msg)
        state["ranked_papers"] = papers
        state["selected_papers"] = papers
        
    return state


def analyze_papers_node(state: ResearchState) -> ResearchState:
    """
    Stage 4: Paper Analysis Node
    Ingests selected papers into RAG and extracts structured analytical fields.
    """
    logger.info("Node [4/6]: Paper Analysis")
    state["status"] = "Ingesting papers into RAG engine & extracting insights..."
    
    selected = state.get("selected_papers", [])
    if not selected:
        state["paper_analysis"] = []
        state["analyses"] = []
        return state

    try:
        rag = RAGPipeline()
        summarizer = Summarizer()
        analyzer = AgentAnalyzer(rag, summarizer)
        
        pdf_paths = [p["local_path"] for p in selected if p.get("local_path") and os.path.exists(p["local_path"])]
        
        if pdf_paths:
            logger.info(f"Ingesting {len(pdf_paths)} paper PDF(s) into FAISS Vector DB...")
            rag.ingest_and_build_index(pdf_paths, force_rebuild=True)
        
        analyses = []
        all_chunks = rag.retriever.vector_db.chunks if hasattr(rag.retriever.vector_db, 'chunks') else []
        
        for paper in selected:
            title = paper.get("title", "Unknown Title")
            source_name = f"{paper.get('id', 'unknown')}.pdf"
            paper_chunks = [c for c in all_chunks if c.get("source") == source_name]
            
            try:
                analysis = analyzer.analyze_paper(title, paper_chunks, source_name)
                analyses.append(analysis)
            except Exception as paper_exc:
                logger.error(f"Error analyzing paper '{title}': {paper_exc}")
                state["errors"].append(f"Analysis error for '{title}': {str(paper_exc)}")
                
        state["paper_analysis"] = analyses
        state["analyses"] = analyses
    except Exception as exc:
        err_msg = f"Global Paper Analysis Error: {str(exc)}"
        logger.error(err_msg)
        state["errors"].append(err_msg)
        state["paper_analysis"] = []
        state["analyses"] = []
        
    return state


def compare_papers_node(state: ResearchState) -> ResearchState:
    """
    Stage 5: Multi-Paper Comparison Node
    Generates a structured comparison matrix across all analyzed papers.
    """
    logger.info("Node [5/6]: Multi-Paper Comparison")
    state["status"] = "Generating multi-paper comparative matrix..."
    
    analyses = state.get("paper_analysis") or state.get("analyses", [])
    try:
        analyzer = AgentAnalyzer(None, None)
        comp = analyzer.compare_papers(analyses)
        state["comparison"] = comp
    except Exception as exc:
        err_msg = f"Comparison Generation Error: {str(exc)}"
        logger.error(err_msg)
        state["errors"].append(err_msg)
        state["comparison"] = "Unable to generate comparison matrix."
        
    return state


def review_papers_node(state: ResearchState) -> ResearchState:
    """
    Stage 6: Literature Review Node
    Synthesizes a grounded literature review document from paper analyses.
    """
    logger.info("Node [6/6]: Literature Review Synthesis")
    state["status"] = "Synthesizing multi-paper literature review..."
    
    topic = state.get("research_topic") or state.get("topic", "")
    analyses = state.get("paper_analysis") or state.get("analyses", [])
    
    try:
        analyzer = AgentAnalyzer(None, None)
        review = analyzer.generate_literature_review(topic, analyses)
        state["literature_review"] = review
        state["status"] = "Agentic Research Analysis Complete."
    except Exception as exc:
        err_msg = f"Literature Review Error: {str(exc)}"
        logger.error(err_msg)
        state["errors"].append(err_msg)
        state["literature_review"] = "Unable to generate literature review."
        state["status"] = "Workflow Completed with Errors."
        
    return state


# ── Graph Construction ───────────────────────────────────────

def build_research_graph():
    """
    Compiles the 6-stage LangGraph workflow.
    """
    workflow = StateGraph(ResearchState)
    
    # Register logical nodes
    workflow.add_node("plan", plan_research_node)
    workflow.add_node("search", search_papers_node)
    workflow.add_node("rank", rank_papers_node)
    workflow.add_node("analyze", analyze_papers_node)
    workflow.add_node("compare", compare_papers_node)
    workflow.add_node("review", review_papers_node)
    
    # Define linear execution flow
    workflow.set_entry_point("plan")
    workflow.add_edge("plan", "search")
    workflow.add_edge("search", "rank")
    workflow.add_edge("rank", "analyze")
    workflow.add_edge("analyze", "compare")
    workflow.add_edge("compare", "review")
    workflow.add_edge("review", END)
    
    return workflow.compile()


class ResearchAgent:
    """
    High-level runner interface for the Agentic AI Research Assistant.
    """
    def __init__(self):
        self.graph = build_research_graph()
        
    def run(self, topic: str) -> dict:
        initial_state = ResearchState(
            research_topic=topic,
            topic=topic,
            research_questions=[],
            discovered_papers=[],
            raw_papers=[],
            ranked_papers=[],
            selected_papers=[],
            paper_analysis=[],
            analyses=[],
            comparison="",
            literature_review="",
            status="Initiating Research Workflow...",
            errors=[]
        )
        logger.info(f"Starting 6-stage Research Workflow for topic: '{topic}'")
        final_state = self.graph.invoke(initial_state)
        return final_state


In [ ]:
%%writefile src/analyzer.py
# ============================================================
# src/analyzer.py
# Agentic Workflow: Structured Analysis, Comparison & Review
# Phase 5, 6, 7 — Pydantic Insight Extractor & Synthesis Engine
# ============================================================

import os
import json
import logging
from pydantic import BaseModel, Field
from typing import List, Dict, Any, Union, Optional

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import LOG_LEVEL
from src.rag_pipeline import RAGPipeline
from src.summarizer import Summarizer

logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


# ── Structured Data Schema ───────────────────────────────────

class PaperAnalysis(BaseModel):
    title: str = Field(description="Title of the research paper")
    research_problem: str = Field(default="Not explicitly mentioned in the paper.")
    objective: str = Field(default="Not explicitly mentioned in the paper.")
    proposed_method: str = Field(default="Not explicitly mentioned in the paper.")
    dataset: str = Field(default="Not explicitly mentioned in the paper.")
    evaluation_metrics: str = Field(default="Not explicitly mentioned in the paper.")
    experimental_results: str = Field(default="Not explicitly mentioned in the paper.")
    key_findings: Union[str, List[str]] = Field(default="Not explicitly mentioned in the paper.")
    limitations: Union[str, List[str]] = Field(default="Not explicitly mentioned in the paper.")
    future_work: Union[str, List[str]] = Field(default="Not explicitly mentioned in the paper.")


class AgentAnalyzer:
    """
    Analyzes individual research papers using the existing VT RAG Pipeline
    and generates multi-paper comparative matrices and literature reviews.
    """
    def __init__(self, rag_pipeline: Optional[RAGPipeline], summarizer: Optional[Summarizer]):
        self.rag = rag_pipeline
        self.summarizer = summarizer

    def _sanitize_field(self, val: Any) -> str:
        """Helper to ensure clean, non-empty text string."""
        if not val or val is None:
            return "Not explicitly mentioned in the paper."
        text = str(val).strip()
        if not text or "cannot answer" in text.lower() or "not found" in text.lower() or "error" in text.lower():
            return "Not explicitly mentioned in the paper."
        return text

    def analyze_paper(self, paper_title: str, chunks: list, source_name: str) -> dict:
        """
        Phase 5: Extracts structured fields from paper text using the existing RAG pipeline.
        """
        logger.info(f"Extracting structured analysis for: '{paper_title}'")
        
        def safe_ask(question: str) -> str:
            q = f"In the paper '{paper_title}', {question}"
            try:
                if not self.rag:
                    return "Not explicitly mentioned in the paper."
                res = self.rag.answer(q, top_k=3)
                ans = res.get("answer", "")
                return self._sanitize_field(ans)
            except Exception as e:
                logger.error(f"QA extraction error for query '{question}': {e}")
                return "Not explicitly mentioned in the paper."

        problem = safe_ask("what is the main research problem being addressed?")
        objective = safe_ask("what is the primary objective or goal?")
        method = safe_ask("what is the proposed method, model, algorithm, or architecture?")
        dataset = safe_ask("what datasets were used for experiments or evaluation?")
        metrics = safe_ask("what evaluation metrics were used?")
        results = safe_ask("what were the main experimental results, findings, or performance numbers?")

        # Use Summarizer for qualitative insights
        insights = {}
        if self.summarizer and chunks:
            try:
                insights = self.summarizer.full_analysis(chunks, source=source_name)
            except Exception as sum_err:
                logger.error(f"Summarizer error for '{source_name}': {sum_err}")

        key_findings = insights.get("key_findings") or "Not explicitly mentioned in the paper."
        limitations = insights.get("limitations") or "Not explicitly mentioned in the paper."
        future_work = insights.get("future_work") or "Not explicitly mentioned in the paper."

        analysis = PaperAnalysis(
            title=paper_title,
            research_problem=problem,
            objective=objective,
            proposed_method=method,
            dataset=dataset,
            evaluation_metrics=metrics,
            experimental_results=results,
            key_findings=key_findings,
            limitations=limitations,
            future_work=future_work
        )
        
        return analysis.model_dump()

    def compare_papers(self, analyses: List[dict]) -> str:
        """
        Phase 6: Generates a multi-paper comparison matrix in Markdown table format.
        """
        logger.info(f"Generating multi-paper comparison matrix for {len(analyses)} papers...")
        if not analyses:
            return "No analyzed papers available for comparison."

        md = "### ⚖️ Multi-Paper Comparative Matrix\n\n"
        md += "| Paper Title | Research Problem | Methodology | Dataset | Metrics & Results | Key Findings | Limitations |\n"
        md += "|---|---|---|---|---|---|---|\n"
        
        for p in analyses:
            title = p.get('title', 'Unknown').replace('|', '-')
            prob = self._sanitize_field(p.get('research_problem')).replace('|', '-').replace('\n', ' ')
            method = self._sanitize_field(p.get('proposed_method')).replace('|', '-').replace('\n', ' ')
            dataset = self._sanitize_field(p.get('dataset')).replace('|', '-').replace('\n', ' ')
            results = self._sanitize_field(p.get('experimental_results')).replace('|', '-').replace('\n', ' ')
            
            findings = p.get('key_findings', '')
            if isinstance(findings, list):
                findings = findings[0] if findings else "Not explicitly mentioned in the paper."
            findings = self._sanitize_field(findings).replace('|', '-').replace('\n', ' ')

            limits = p.get('limitations', '')
            if isinstance(limits, list):
                limits = limits[0] if limits else "Not explicitly mentioned in the paper."
            limits = self._sanitize_field(limits).replace('|', '-').replace('\n', ' ')

            md += f"| **{title}** | {prob[:120]} | {method[:120]} | {dataset[:90]} | {results[:120]} | {findings[:120]} | {limits[:90]} |\n"

        return md

    def generate_literature_review(self, topic: str, analyses: List[dict]) -> str:
        """
        Phase 7: Synthesizes a structured 6-section literature review grounded strictly in paper analyses.
        """
        logger.info(f"Synthesizing literature review for topic: '{topic}'...")
        if not analyses:
            return "No papers available to generate literature review."

        review = f"# 📝 Literature Review: {topic}\n\n"
        
        # 1. Introduction
        review += "## 1. Introduction\n"
        review += f"This literature review provides a structured synthesis of **{len(analyses)}** peer-reviewed scholarly papers addressing the topic of **{topic}**. "
        review += "The reviewed studies focus on advancing domain-specific solutions, evaluating model performance, and highlighting current research challenges.\n\n"

        # 2. Existing Approaches
        review += "## 2. Existing Approaches\n"
        for p in analyses:
            title = p.get("title", "Unknown")
            prob = self._sanitize_field(p.get("research_problem"))
            obj = self._sanitize_field(p.get("objective"))
            review += f"- **{title}**: Investigates the problem of *{prob}* with the primary objective to *{obj}*.\n"
        review += "\n"

        # 3. Methodology Comparison
        review += "## 3. Methodology Comparison\n"
        for p in analyses:
            title = p.get("title", "Unknown")
            method = self._sanitize_field(p.get("proposed_method"))
            review += f"- **{title}**: Employs **{method}** as its core framework.\n"
        review += "\n"

        # 4. Major Findings & Performance
        review += "## 4. Major Findings & Performance\n"
        for p in analyses:
            title = p.get("title", "Unknown")
            dataset = self._sanitize_field(p.get("dataset"))
            metrics = self._sanitize_field(p.get("evaluation_metrics"))
            results = self._sanitize_field(p.get("experimental_results"))
            review += f"- **{title}**: Evaluated on **{dataset}** using **{metrics}**. Experimental outcomes demonstrated: *{results}*.\n"
        review += "\n"

        # 5. Common Limitations
        review += "## 5. Common Limitations\n"
        for p in analyses:
            title = p.get("title", "Unknown")
            limits = p.get("limitations", "Not explicitly mentioned in the paper.")
            if isinstance(limits, list):
                limits = "; ".join(str(x) for x in limits if x)
            review += f"- **{title}**: {self._sanitize_field(limits)}\n"
        review += "\n"

        # 6. Future Directions
        review += "## 6. Future Directions\n"
        review += "Across the analyzed literature, key avenues for future investigation include:\n"
        for p in analyses:
            title = p.get("title", "Unknown")
            fw = p.get("future_work", "Not explicitly mentioned in the paper.")
            if isinstance(fw, list):
                fw = "; ".join(str(x) for x in fw if x)
            review += f"- **{title}**: {self._sanitize_field(fw)}\n"
        review += "\n---\n*Synthesis generated autonomously by the Agentic AI Research Assistant.*"

        return review


In [ ]:
%%writefile src/citation_graph.py
# ============================================================
# src/citation_graph.py
# Concept Co-occurrence Graph for Paper Relationships
#
# Extracts key concepts from paper abstracts using TF-IDF
# and builds a concept co-occurrence graph showing how
# papers relate to each other through shared concepts.
# ============================================================

import re
import math
import logging
from collections import Counter

logger = logging.getLogger(__name__)


class ConceptGraphBuilder:
    """
    Builds a concept co-occurrence graph from paper abstracts.

    Extracts key concepts using TF-IDF-style scoring and builds
    edges between papers that share significant concepts.

    Usage:
        builder = ConceptGraphBuilder()
        graph = builder.build_graph(papers)
        mermaid = builder.to_mermaid(graph)
    """

    # Common academic stopwords to filter out
    STOPWORDS = {
        "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
        "of", "with", "by", "from", "up", "about", "into", "over", "after",
        "is", "are", "was", "were", "be", "been", "being", "have", "has",
        "had", "do", "does", "did", "will", "would", "could", "should",
        "may", "might", "shall", "can", "this", "that", "these", "those",
        "it", "its", "we", "our", "they", "their", "them", "which", "who",
        "whom", "what", "where", "when", "how", "than", "then", "also",
        "not", "no", "nor", "so", "if", "as", "such", "each", "every",
        "all", "both", "few", "more", "most", "other", "some", "any",
        "new", "used", "using", "based", "paper", "propose", "proposed",
        "method", "approach", "results", "show", "work", "two", "one",
        "first", "use", "however", "well", "still", "even", "also",
        "between", "through", "during", "before", "while", "across",
        "several", "many", "different", "various", "existing", "recent",
    }

    def __init__(self, top_k_concepts: int = 8, min_shared: int = 2):
        """
        Args:
            top_k_concepts: Number of key concepts to extract per paper.
            min_shared: Minimum shared concepts for an edge between papers.
        """
        self.top_k_concepts = top_k_concepts
        self.min_shared = min_shared

    def _extract_ngrams(self, text: str) -> list:
        """Extract unigrams and bigrams from text."""
        # Lowercase and clean
        text = text.lower()
        text = re.sub(r'[^a-z\s\-]', ' ', text)
        words = [w for w in text.split() if len(w) > 2 and w not in self.STOPWORDS]

        # Unigrams
        ngrams = list(words)

        # Bigrams (joined with space)
        for i in range(len(words) - 1):
            bigram = f"{words[i]} {words[i+1]}"
            ngrams.append(bigram)

        return ngrams

    def _tfidf_keywords(self, abstracts: list) -> list:
        """
        Extract top keywords from each abstract using TF-IDF scoring.

        Args:
            abstracts: List of abstract strings.

        Returns:
            List of sets, each set containing top concepts for that paper.
        """
        # Build document frequency
        doc_ngrams = []
        doc_freq = Counter()

        for abstract in abstracts:
            ngrams = self._extract_ngrams(abstract)
            unique = set(ngrams)
            doc_ngrams.append(Counter(ngrams))
            for ng in unique:
                doc_freq[ng] += 1

        n_docs = len(abstracts)
        results = []

        for tf_counts in doc_ngrams:
            # Compute TF-IDF for each ngram
            scores = {}
            total = sum(tf_counts.values())
            for ngram, count in tf_counts.items():
                tf = count / total
                idf = math.log(n_docs / (1 + doc_freq[ngram]))
                # Boost bigrams
                boost = 1.5 if ' ' in ngram else 1.0
                scores[ngram] = tf * idf * boost

            # Take top-k
            top = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:self.top_k_concepts]
            results.append(set(kw for kw, _ in top))

        return results

    def build_graph(self, papers: list) -> dict:
        """
        Build a concept co-occurrence graph from papers.

        Args:
            papers: List of paper dicts with 'title' and 'abstract'.

        Returns:
            {
                "nodes": [{"id": str, "title": str, "concepts": list}],
                "edges": [{"source": str, "target": str, "shared": list, "weight": int}],
            }
        """
        if not papers:
            return {"nodes": [], "edges": []}

        abstracts = [p.get("abstract", "") for p in papers]
        paper_concepts = self._tfidf_keywords(abstracts)

        # Build nodes
        nodes = []
        for i, paper in enumerate(papers):
            short_title = paper.get("title", f"Paper {i+1}")
            if len(short_title) > 40:
                short_title = short_title[:37] + "..."
            nodes.append({
                "id": f"P{i+1}",
                "title": short_title,
                "concepts": sorted(paper_concepts[i]),
            })

        # Build edges
        edges = []
        for i in range(len(papers)):
            for j in range(i + 1, len(papers)):
                shared = paper_concepts[i] & paper_concepts[j]
                if len(shared) >= self.min_shared:
                    edges.append({
                        "source": f"P{i+1}",
                        "target": f"P{j+1}",
                        "shared": sorted(shared),
                        "weight": len(shared),
                    })

        logger.info(f"Concept graph: {len(nodes)} nodes, {len(edges)} edges")
        return {"nodes": nodes, "edges": edges}

    def to_mermaid(self, graph: dict) -> str:
        """
        Convert the graph to a Mermaid diagram string.

        Args:
            graph: Graph dict from build_graph().

        Returns:
            Mermaid diagram string.
        """
        if not graph["nodes"]:
            return "graph LR\n    empty[No papers to visualize]"

        lines = ["graph LR"]

        # Style definitions
        lines.append("    classDef paper fill:#1a1a2e,stroke:#00d4ff,stroke-width:2px,color:#e0e0e0")
        lines.append("    classDef concept fill:#0f0f23,stroke:#a855f7,stroke-width:1px,color:#c0c0c0,font-size:10px")

        # Node definitions
        for node in graph["nodes"]:
            safe_title = node["title"].replace('"', "'")
            lines.append(f'    {node["id"]}["{safe_title}"]')

        # Edges with shared concept labels
        for edge in graph["edges"]:
            label = ", ".join(edge["shared"][:3])
            if len(edge["shared"]) > 3:
                label += f" +{len(edge['shared'])-3}"
            safe_label = label.replace('"', "'")
            thickness = "==>" if edge["weight"] >= 3 else "-->"
            lines.append(f'    {edge["source"]} {thickness}|"{safe_label}"| {edge["target"]}')

        # Apply styles
        node_ids = " & ".join(n["id"] for n in graph["nodes"])
        if node_ids:
            lines.append(f"    class {node_ids} paper")

        return "\n".join(lines)

    def to_summary(self, graph: dict) -> str:
        """
        Generate a text summary of the concept relationships.

        Args:
            graph: Graph dict from build_graph().

        Returns:
            Human-readable summary string.
        """
        if not graph["edges"]:
            return "No significant concept overlap found between papers."

        lines = ["**Key Concept Relationships:**\n"]
        for edge in sorted(graph["edges"], key=lambda e: e["weight"], reverse=True):
            src = next(n for n in graph["nodes"] if n["id"] == edge["source"])
            tgt = next(n for n in graph["nodes"] if n["id"] == edge["target"])
            concepts = ", ".join(edge["shared"])
            lines.append(f"- **{src['title']}** ↔ **{tgt['title']}** share: _{concepts}_")

        return "\n".join(lines)


In [ ]:
%%writefile src/export.py
# ============================================================
# src/export.py
# PDF Report Export for Research Workflow Results
#
# Uses fpdf2 to generate formatted PDF reports from the
# agentic workflow results.
# ============================================================

import os
import logging
from datetime import datetime

logger = logging.getLogger(__name__)

try:
    from fpdf import FPDF
except ImportError:
    FPDF = None
    logger.warning("fpdf2 not installed. PDF export disabled. Run: pip install fpdf2")


class ResearchReportPDF:
    """
    Generates a formatted PDF report from agentic workflow results.

    Usage:
        exporter = ResearchReportPDF()
        pdf_bytes = exporter.generate(topic, final_state)
    """

    def __init__(self):
        if FPDF is None:
            raise ImportError("fpdf2 is required for PDF export. Install with: pip install fpdf2")

    def generate(self, topic: str, state: dict) -> bytes:
        """
        Generate a complete PDF report from the workflow state.

        Args:
            topic: The research topic.
            state: The final state dict from the agentic workflow.

        Returns:
            PDF content as bytes.
        """
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=20)

        # ── Cover Page ──
        pdf.add_page()
        pdf.set_font("Helvetica", "B", 28)
        pdf.ln(40)
        pdf.cell(0, 15, "Research Report", new_x="LMARGIN", new_y="NEXT", align="C")
        pdf.set_font("Helvetica", "", 14)
        pdf.ln(5)
        pdf.set_text_color(100, 100, 100)
        pdf.cell(0, 10, "Generated by Research Paper Intelligence Engine", new_x="LMARGIN", new_y="NEXT", align="C")
        pdf.ln(20)
        pdf.set_text_color(0, 0, 0)
        pdf.set_font("Helvetica", "B", 16)
        pdf.multi_cell(0, 10, f"Topic: {topic}", align="C")
        pdf.ln(10)
        pdf.set_font("Helvetica", "", 11)
        pdf.set_text_color(120, 120, 120)
        pdf.cell(0, 10, f"Date: {datetime.now().strftime('%B %d, %Y at %H:%M')}", new_x="LMARGIN", new_y="NEXT", align="C")
        num_papers = len(state.get("selected_papers", []))
        pdf.cell(0, 8, f"Papers Analyzed: {num_papers}", new_x="LMARGIN", new_y="NEXT", align="C")

        # ── Top Ranked Papers ──
        pdf.add_page()
        self._section_header(pdf, "Top Ranked Papers")

        for i, paper in enumerate(state.get("selected_papers", []), start=1):
            title = paper.get("title", "Unknown")
            year = paper.get("year", "N/A")
            score = paper.get("relevance_score", 0)
            authors = ", ".join(paper.get("authors", [])[:3])
            if len(paper.get("authors", [])) > 3:
                authors += " et al."

            pdf.set_font("Helvetica", "B", 11)
            pdf.set_text_color(0, 100, 200)
            pdf.cell(0, 7, f"{i}. {self._safe(title)}", new_x="LMARGIN", new_y="NEXT")
            pdf.set_text_color(0, 0, 0)
            pdf.set_font("Helvetica", "", 9)
            pdf.cell(0, 5, f"   Year: {year}  |  Relevance Score: {score:.3f}", new_x="LMARGIN", new_y="NEXT")
            if authors:
                pdf.cell(0, 5, f"   Authors: {self._safe(authors)}", new_x="LMARGIN", new_y="NEXT")

            abstract = paper.get("abstract", "")
            if abstract:
                pdf.set_font("Helvetica", "I", 8)
                pdf.set_text_color(80, 80, 80)
                pdf.multi_cell(0, 4, f"   {self._safe(abstract[:300])}...")
                pdf.set_text_color(0, 0, 0)
            pdf.ln(4)

        # ── Structured Analysis ──
        analyses = state.get("analyses", [])
        if analyses:
            pdf.add_page()
            self._section_header(pdf, "Structured Analysis")

            for p in analyses:
                pdf.set_font("Helvetica", "B", 11)
                pdf.set_text_color(0, 100, 200)
                pdf.cell(0, 8, self._safe(p.get("title", "Unknown")), new_x="LMARGIN", new_y="NEXT")
                pdf.set_text_color(0, 0, 0)

                fields = [
                    ("Research Problem", p.get("research_problem", "")),
                    ("Objective", p.get("objective", "")),
                    ("Proposed Method", p.get("proposed_method", "")),
                    ("Dataset", p.get("dataset", "")),
                    ("Evaluation Metrics", p.get("evaluation_metrics", "")),
                    ("Results", p.get("experimental_results", "")),
                ]

                for label, value in fields:
                    if value and value != "Not explicitly stated.":
                        pdf.set_font("Helvetica", "B", 9)
                        pdf.cell(40, 5, f"  {label}:", new_x="END")
                        pdf.set_font("Helvetica", "", 9)
                        pdf.multi_cell(0, 5, f" {self._safe(str(value)[:200])}")

                # Key Findings
                findings = p.get("key_findings", [])
                if findings:
                    pdf.set_font("Helvetica", "B", 9)
                    pdf.cell(0, 6, "  Key Findings:", new_x="LMARGIN", new_y="NEXT")
                    pdf.set_font("Helvetica", "", 8)
                    if isinstance(findings, list):
                        for f in findings[:3]:
                            pdf.multi_cell(0, 4, f"    - {self._safe(str(f)[:150])}")
                    else:
                        pdf.multi_cell(0, 4, f"    {self._safe(str(findings)[:300])}")

                pdf.ln(6)
                pdf.set_draw_color(200, 200, 200)
                pdf.line(pdf.get_x() + 10, pdf.get_y(), pdf.get_x() + 180, pdf.get_y())
                pdf.ln(4)

        # ── Multi-Paper Comparison Table ──
        if analyses:
            pdf.add_page("L")  # Landscape for table
            self._section_header(pdf, "Multi-Paper Comparison")

            # Table header
            col_widths = [60, 55, 50, 55, 55]
            headers = ["Paper", "Method", "Dataset", "Key Findings", "Limitations"]

            pdf.set_font("Helvetica", "B", 8)
            pdf.set_fill_color(230, 240, 250)
            for i, header in enumerate(headers):
                pdf.cell(col_widths[i], 7, header, border=1, fill=True, align="C")
            pdf.ln()

            # Table rows
            pdf.set_font("Helvetica", "", 7)
            for p in analyses:
                title = self._safe(p.get("title", "")[:40])
                method = self._safe(str(p.get("proposed_method", ""))[:40])
                dataset = self._safe(str(p.get("dataset", ""))[:35])

                findings = p.get("key_findings", "")
                if isinstance(findings, list):
                    findings = findings[0] if findings else ""
                findings = self._safe(str(findings)[:40])

                limitations = p.get("limitations", "")
                if isinstance(limitations, list):
                    limitations = limitations[0] if limitations else ""
                limitations = self._safe(str(limitations)[:40])

                row_data = [title, method, dataset, findings, limitations]
                for i, cell_text in enumerate(row_data):
                    pdf.cell(col_widths[i], 6, cell_text, border=1)
                pdf.ln()

        # ── Literature Review ──
        lit_review = state.get("literature_review", "")
        if lit_review:
            pdf.add_page("P")  # Back to portrait
            self._section_header(pdf, "Literature Review")
            pdf.set_font("Helvetica", "", 10)
            # Parse markdown-style sections
            for line in lit_review.split("\n"):
                line = line.strip()
                if not line:
                    pdf.ln(3)
                    continue
                if line.startswith("## "):
                    pdf.set_font("Helvetica", "B", 13)
                    pdf.set_text_color(0, 80, 160)
                    pdf.cell(0, 8, self._safe(line[3:]), new_x="LMARGIN", new_y="NEXT")
                    pdf.set_text_color(0, 0, 0)
                elif line.startswith("### "):
                    pdf.set_font("Helvetica", "B", 11)
                    pdf.set_text_color(80, 40, 120)
                    pdf.cell(0, 7, self._safe(line[4:]), new_x="LMARGIN", new_y="NEXT")
                    pdf.set_text_color(0, 0, 0)
                elif line.startswith("- "):
                    pdf.set_font("Helvetica", "", 9)
                    pdf.multi_cell(0, 5, f"  \u2022 {self._safe(line[2:])}")
                elif line.startswith("---"):
                    pdf.ln(3)
                    pdf.set_draw_color(180, 180, 180)
                    pdf.line(pdf.get_x() + 10, pdf.get_y(), pdf.get_x() + 180, pdf.get_y())
                    pdf.ln(3)
                else:
                    pdf.set_font("Helvetica", "", 9)
                    pdf.multi_cell(0, 5, self._safe(line))

        # ── Footer on all pages ──
        # fpdf2 doesn't have a built-in footer callback like reportlab,
        # so we just add a note at the end
        pdf.ln(10)
        pdf.set_font("Helvetica", "I", 8)
        pdf.set_text_color(150, 150, 150)
        pdf.cell(0, 5, "Generated by Research Paper Intelligence Engine | Agentic AI Research Assistant", align="C")

        return pdf.output()

    def _section_header(self, pdf, title: str):
        """Add a styled section header."""
        pdf.set_font("Helvetica", "B", 16)
        pdf.set_text_color(0, 80, 160)
        pdf.cell(0, 12, title, new_x="LMARGIN", new_y="NEXT")
        pdf.set_draw_color(0, 150, 220)
        pdf.set_line_width(0.5)
        pdf.line(pdf.get_x(), pdf.get_y(), pdf.get_x() + 190, pdf.get_y())
        pdf.ln(6)
        pdf.set_text_color(0, 0, 0)
        pdf.set_line_width(0.2)

    def _safe(self, text: str) -> str:
        """Make text safe for fpdf2 (handle encoding issues)."""
        if not text:
            return ""
        # Replace characters that fpdf2 can't encode in latin-1
        return text.encode("latin-1", errors="replace").decode("latin-1")


In [ ]:
%%writefile app.py
# ============================================================
# app.py
# Phase 9 — Streamlit UI (Premium Animated Edition)
#
# Research Paper Intelligence Engine
# A complete web interface for uploading PDFs, asking
# questions via RAG, generating summaries, and extracting
# structured research insights.
# ============================================================

import os
import sys
import tempfile
import logging
import time

# Move HuggingFace cache to E: drive immediately because C: drive is full
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
os.environ["HF_HOME"] = os.path.join(BASE_DIR, "hf_cache")

# Suppress harmless torchvision/transformers warnings from Streamlit file watcher
os.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import streamlit as st

# ── Path setup (so src/ is importable) ───────────────────────
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, BASE_DIR)

from config import TOP_K_RESULTS
from src.pdf_processor import PDFProcessor
from src.chunking      import TextChunker
from src.embeddings    import EmbeddingGenerator
from src.vector_db     import VectorDB
from src.retriever     import Retriever
from src.rag_pipeline  import RAGPipeline
from src.summarizer    import Summarizer
from src.agent         import ResearchAgent

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================================
# Streamlit Page Configuration
# ============================================================
st.set_page_config(
    page_title  = "Research Paper Intelligence Engine",
    page_icon   = "🧠",
    layout      = "wide",
    initial_sidebar_state = "expanded",
)

# ============================================================
# Custom CSS — Premium Animated Dark Theme
# ============================================================
st.markdown("""
<style>
/* ── Google Fonts ── */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500;600&display=swap');

/* ── Root Variables ── */
:root {
    --bg-primary    : #0a0a1a;
    --bg-secondary  : #0f0f23;
    --bg-card       : rgba(20, 20, 50, 0.6);
    --bg-glass      : rgba(255, 255, 255, 0.03);
    --accent-cyan   : #00d4ff;
    --accent-purple : #a855f7;
    --accent-green  : #22c55e;
    --accent-amber  : #f59e0b;
    --accent-rose   : #f43f5e;
    --accent-blue   : #3b82f6;
    --text-primary  : #e2e8f0;
    --text-secondary: #94a3b8;
    --text-muted    : #64748b;
    --border-glass  : rgba(255, 255, 255, 0.08);
    --border-glow   : rgba(0, 212, 255, 0.3);
    --radius        : 16px;
    --radius-sm     : 10px;
    --shadow-glow   : 0 0 30px rgba(0, 212, 255, 0.1);
    --transition    : all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
}

/* ── Keyframe Animations ── */
@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(24px); }
    to   { opacity: 1; transform: translateY(0); }
}

@keyframes fadeInDown {
    from { opacity: 0; transform: translateY(-16px); }
    to   { opacity: 1; transform: translateY(0); }
}

@keyframes shimmer {
    0%   { background-position: -200% center; }
    100% { background-position: 200% center; }
}

@keyframes pulseGlow {
    0%, 100% { box-shadow: 0 0 10px rgba(0, 212, 255, 0.2); }
    50%      { box-shadow: 0 0 25px rgba(0, 212, 255, 0.5); }
}

@keyframes borderGlow {
    0%, 100% { border-color: rgba(0, 212, 255, 0.2); }
    50%      { border-color: rgba(0, 212, 255, 0.6); }
}

@keyframes gradientShift {
    0%   { background-position: 0% 50%; }
    50%  { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

@keyframes float {
    0%, 100% { transform: translateY(0px); }
    50%      { transform: translateY(-8px); }
}

@keyframes blink {
    0%, 100% { opacity: 1; }
    50%      { opacity: 0; }
}

@keyframes spinSlow {
    from { transform: rotate(0deg); }
    to   { transform: rotate(360deg); }
}

@keyframes progressStripe {
    0%   { background-position: 0 0; }
    100% { background-position: 40px 0; }
}

@keyframes nodeActivePulse {
    0%, 100% { transform: scale(1); box-shadow: 0 0 10px rgba(0, 212, 255, 0.3); }
    50%      { transform: scale(1.08); box-shadow: 0 0 25px rgba(0, 212, 255, 0.7); }
}

@keyframes slideInLeft {
    from { opacity: 0; transform: translateX(-20px); }
    to   { opacity: 1; transform: translateX(0); }
}

@keyframes typewriter {
    from { width: 0; }
    to   { width: 100%; }
}

/* ── Scrollbar ── */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: var(--bg-primary); }
::-webkit-scrollbar-thumb { background: var(--accent-cyan); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--accent-purple); }

/* ── Base ── */
html, body, [data-testid="stAppViewContainer"] {
    background: var(--bg-primary) !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif;
    color: var(--text-primary);
}

[data-testid="stAppViewContainer"] > section > div {
    background: var(--bg-primary) !important;
}

[data-testid="stHeader"] {
    background: rgba(10, 10, 26, 0.8) !important;
    backdrop-filter: blur(12px);
}

/* ── Sidebar ── */
[data-testid="stSidebar"] {
    background: linear-gradient(180deg, #0d0d24 0%, #0a0a1a 100%) !important;
    border-right: 1px solid var(--border-glass);
}

[data-testid="stSidebar"]::before {
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    height: 3px;
    background: linear-gradient(90deg, var(--accent-cyan), var(--accent-purple), var(--accent-green));
    background-size: 200% 100%;
    animation: gradientShift 4s ease infinite;
    z-index: 999;
}

/* ── Typography ── */
h1, h2, h3, h4 {
    font-family: 'Inter', sans-serif;
    font-weight: 700;
    color: var(--text-primary) !important;
    letter-spacing: -0.02em;
}
h1 { font-size: 1.8rem !important; }
h2 { font-size: 1.35rem !important; color: var(--accent-cyan) !important; }
h3 { font-size: 1.15rem !important; color: var(--accent-purple) !important; }

/* ── Glass Card ── */
.glass-card {
    background: var(--bg-card);
    backdrop-filter: blur(16px);
    border: 1px solid var(--border-glass);
    border-radius: var(--radius);
    padding: 1.5rem;
    margin-bottom: 1rem;
    animation: fadeInUp 0.5s ease-out both;
    transition: var(--transition);
    position: relative;
    overflow: hidden;
}
.glass-card::before {
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    height: 2px;
    background: linear-gradient(90deg, transparent, var(--accent-cyan), transparent);
    opacity: 0;
    transition: opacity 0.3s;
}
.glass-card:hover {
    border-color: var(--border-glow);
    box-shadow: var(--shadow-glow);
    transform: translateY(-2px);
}
.glass-card:hover::before {
    opacity: 1;
}

/* Staggered animation delays */
.glass-card:nth-child(1) { animation-delay: 0.05s; }
.glass-card:nth-child(2) { animation-delay: 0.1s; }
.glass-card:nth-child(3) { animation-delay: 0.15s; }
.glass-card:nth-child(4) { animation-delay: 0.2s; }
.glass-card:nth-child(5) { animation-delay: 0.25s; }

/* ── Animated Header ── */
.app-header {
    text-align: center;
    padding: 2rem 1rem 1.5rem;
    margin-bottom: 1.5rem;
    position: relative;
    animation: fadeInDown 0.6s ease-out;
}
.app-header h1 {
    font-size: 2.2rem !important;
    font-weight: 800 !important;
    background: linear-gradient(135deg, var(--accent-cyan), var(--accent-purple), var(--accent-green), var(--accent-cyan));
    background-size: 300% 300%;
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    animation: gradientShift 5s ease infinite;
    margin-bottom: 0.5rem;
    letter-spacing: -0.03em;
}
.app-subtitle {
    color: var(--text-secondary);
    font-size: 1rem;
    font-weight: 400;
    max-width: 640px;
    margin: 0 auto;
    line-height: 1.6;
}
.header-decoration {
    display: flex;
    justify-content: center;
    gap: 8px;
    margin-top: 1rem;
}
.header-dot {
    width: 6px;
    height: 6px;
    border-radius: 50%;
    animation: float 3s ease-in-out infinite;
}
.header-dot:nth-child(1) { background: var(--accent-cyan); animation-delay: 0s; }
.header-dot:nth-child(2) { background: var(--accent-purple); animation-delay: 0.3s; }
.header-dot:nth-child(3) { background: var(--accent-green); animation-delay: 0.6s; }
.header-dot:nth-child(4) { background: var(--accent-amber); animation-delay: 0.9s; }
.header-dot:nth-child(5) { background: var(--accent-rose); animation-delay: 1.2s; }

/* ── Pipeline Progress ── */
.pipeline-container {
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 0;
    padding: 2rem 1rem;
    margin: 1.5rem 0;
    animation: fadeInUp 0.5s ease-out;
}
.pipeline-node {
    display: flex;
    flex-direction: column;
    align-items: center;
    gap: 8px;
    position: relative;
    z-index: 2;
}
.pipeline-icon {
    width: 52px;
    height: 52px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 1.3rem;
    border: 2px solid var(--border-glass);
    background: var(--bg-secondary);
    transition: var(--transition);
    position: relative;
}
.pipeline-icon.pending {
    opacity: 0.4;
    border-color: var(--text-muted);
}
.pipeline-icon.active {
    border-color: var(--accent-cyan);
    background: rgba(0, 212, 255, 0.1);
    animation: nodeActivePulse 2s ease-in-out infinite;
}
.pipeline-icon.done {
    border-color: var(--accent-green);
    background: rgba(34, 197, 94, 0.15);
    box-shadow: 0 0 15px rgba(34, 197, 94, 0.3);
}
.pipeline-label {
    font-size: 0.72rem;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    color: var(--text-muted);
    transition: var(--transition);
    white-space: nowrap;
}
.pipeline-label.active { color: var(--accent-cyan); }
.pipeline-label.done   { color: var(--accent-green); }
.pipeline-connector {
    width: 60px;
    height: 3px;
    background: var(--border-glass);
    position: relative;
    margin: 0 -4px;
    margin-bottom: 22px;
    z-index: 1;
    border-radius: 2px;
    overflow: hidden;
}
.pipeline-connector.done {
    background: var(--accent-green);
    box-shadow: 0 0 8px rgba(34, 197, 94, 0.4);
}
.pipeline-connector.active {
    background: linear-gradient(90deg, var(--accent-green), var(--accent-cyan));
}
.pipeline-connector.active::after {
    content: '';
    position: absolute;
    top: 0;
    left: -100%;
    width: 100%;
    height: 100%;
    background: linear-gradient(90deg, transparent, rgba(0, 212, 255, 0.8), transparent);
    animation: shimmer 1.5s ease-in-out infinite;
}

/* ── Result Cards ── */
.result-paper {
    background: var(--bg-card);
    backdrop-filter: blur(12px);
    border: 1px solid var(--border-glass);
    border-radius: var(--radius-sm);
    padding: 1.2rem 1.5rem;
    margin-bottom: 0.8rem;
    animation: fadeInUp 0.4s ease-out both;
    transition: var(--transition);
    display: flex;
    align-items: flex-start;
    gap: 1rem;
}
.result-paper:hover {
    border-color: var(--border-glow);
    box-shadow: 0 0 20px rgba(0, 212, 255, 0.1);
    transform: translateY(-1px);
}
.paper-rank {
    min-width: 36px;
    height: 36px;
    border-radius: 10px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 700;
    font-size: 0.85rem;
    flex-shrink: 0;
}
.rank-1 { background: linear-gradient(135deg, #f59e0b, #d97706); color: #1a1a2e; }
.rank-2 { background: linear-gradient(135deg, #94a3b8, #64748b); color: #1a1a2e; }
.rank-3 { background: linear-gradient(135deg, #a16207, #92400e); color: #fef3c7; }
.rank-default { background: rgba(100, 116, 139, 0.2); color: var(--text-secondary); border: 1px solid var(--border-glass); }
.paper-info { flex: 1; }
.paper-title {
    font-weight: 600;
    font-size: 0.95rem;
    color: var(--text-primary);
    margin-bottom: 0.3rem;
}
.paper-meta {
    font-size: 0.8rem;
    color: var(--text-muted);
    display: flex;
    align-items: center;
    gap: 0.8rem;
    flex-wrap: wrap;
}
.paper-score-bar {
    width: 80px;
    height: 5px;
    background: rgba(255, 255, 255, 0.05);
    border-radius: 3px;
    overflow: hidden;
}
.paper-score-fill {
    height: 100%;
    border-radius: 3px;
    background: linear-gradient(90deg, var(--accent-cyan), var(--accent-purple));
    transition: width 0.8s ease-out;
}

/* ── Answer Box ── */
.answer-box {
    background: var(--bg-card);
    backdrop-filter: blur(12px);
    border: 1px solid var(--border-glass);
    border-left: 4px solid var(--accent-cyan);
    border-radius: 0 var(--radius-sm) var(--radius-sm) 0;
    padding: 1.5rem;
    font-family: 'Inter', sans-serif;
    font-size: 0.95rem;
    line-height: 1.75;
    margin-bottom: 1rem;
    animation: fadeInUp 0.4s ease-out;
    color: var(--text-primary);
}
.answer-box.streaming {
    animation: borderGlow 2s ease-in-out infinite;
}
.cursor-blink {
    animation: blink 1s step-end infinite;
    color: var(--accent-cyan);
    font-weight: 700;
}

/* ── Insight Cards ── */
.insight-card {
    background: var(--bg-card);
    backdrop-filter: blur(12px);
    border: 1px solid var(--border-glass);
    border-radius: var(--radius-sm);
    padding: 1.2rem;
    margin-bottom: 0.8rem;
    animation: fadeInUp 0.4s ease-out both;
    transition: var(--transition);
    position: relative;
    overflow: hidden;
}
.insight-card:hover {
    transform: translateY(-2px);
    box-shadow: var(--shadow-glow);
}
.insight-card.findings {
    border-left: 4px solid var(--accent-green);
}
.insight-card.findings:hover { box-shadow: 0 0 25px rgba(34, 197, 94, 0.15); }
.insight-card.limits {
    border-left: 4px solid var(--accent-amber);
}
.insight-card.limits:hover { box-shadow: 0 0 25px rgba(245, 158, 11, 0.15); }
.insight-card.future {
    border-left: 4px solid var(--accent-purple);
}
.insight-card.future:hover { box-shadow: 0 0 25px rgba(168, 85, 247, 0.15); }
.insight-card h4 {
    font-size: 0.95rem !important;
    margin-bottom: 0.6rem;
    display: flex;
    align-items: center;
    gap: 0.5rem;
}
.insight-card ul {
    margin: 0;
    padding-left: 1.2rem;
    color: var(--text-secondary);
    font-size: 0.9rem;
    line-height: 1.7;
}
.insight-card li { margin-bottom: 0.3rem; }

/* ── Confidence Badge ── */
.badge {
    display: inline-flex;
    align-items: center;
    gap: 4px;
    padding: 0.15rem 0.6rem;
    border-radius: 20px;
    font-size: 0.7rem;
    font-weight: 600;
    letter-spacing: 0.05em;
    font-family: 'JetBrains Mono', monospace;
}
.badge-synth {
    background: rgba(0, 212, 255, 0.1);
    color: var(--accent-cyan);
    border: 1px solid rgba(0, 212, 255, 0.3);
}

/* ── Source Chips ── */
.source-chip {
    display: inline-flex;
    align-items: center;
    gap: 4px;
    background: rgba(168, 85, 247, 0.1);
    border: 1px solid rgba(168, 85, 247, 0.2);
    border-radius: 20px;
    padding: 0.2rem 0.7rem;
    font-size: 0.78rem;
    color: var(--accent-purple);
    margin: 0.2rem 0.2rem 0.2rem 0;
    font-family: 'JetBrains Mono', monospace;
    transition: var(--transition);
}
.source-chip:hover {
    background: rgba(168, 85, 247, 0.2);
    border-color: rgba(168, 85, 247, 0.5);
    box-shadow: 0 0 12px rgba(168, 85, 247, 0.2);
}

/* ── Buttons ── */
[data-testid="stButton"] > button {
    background: linear-gradient(135deg, rgba(0, 212, 255, 0.15), rgba(168, 85, 247, 0.15)) !important;
    color: var(--text-primary) !important;
    border: 1px solid rgba(0, 212, 255, 0.3) !important;
    border-radius: var(--radius-sm) !important;
    font-family: 'Inter', sans-serif !important;
    font-weight: 600 !important;
    padding: 0.5rem 1.2rem !important;
    transition: var(--transition) !important;
    letter-spacing: 0.01em;
}
[data-testid="stButton"] > button:hover {
    background: linear-gradient(135deg, rgba(0, 212, 255, 0.25), rgba(168, 85, 247, 0.25)) !important;
    border-color: var(--accent-cyan) !important;
    box-shadow: 0 0 20px rgba(0, 212, 255, 0.2) !important;
    transform: translateY(-1px);
}
[data-testid="stButton"] > button:active {
    transform: translateY(0px);
}

/* ── Primary Action Button ── */
.stButton > button[kind="primary"],
.action-btn > button {
    background: linear-gradient(135deg, var(--accent-cyan), var(--accent-purple)) !important;
    border: none !important;
    color: #0a0a1a !important;
    font-weight: 700 !important;
}

/* ── File Uploader ── */
[data-testid="stFileUploader"] {
    border: 2px dashed rgba(0, 212, 255, 0.2) !important;
    border-radius: var(--radius-sm) !important;
    background: rgba(0, 212, 255, 0.03) !important;
    transition: var(--transition);
}
[data-testid="stFileUploader"]:hover {
    border-color: rgba(0, 212, 255, 0.4) !important;
    background: rgba(0, 212, 255, 0.06) !important;
}

/* ── Text Inputs ── */
[data-testid="stTextInput"] input,
[data-testid="stTextArea"] textarea {
    background: rgba(15, 15, 35, 0.8) !important;
    border: 1px solid var(--border-glass) !important;
    border-radius: var(--radius-sm) !important;
    color: var(--text-primary) !important;
    font-family: 'Inter', sans-serif !important;
    transition: var(--transition);
}
[data-testid="stTextInput"] input:focus,
[data-testid="stTextArea"] textarea:focus {
    border-color: var(--accent-cyan) !important;
    box-shadow: 0 0 15px rgba(0, 212, 255, 0.15) !important;
    outline: none !important;
}

/* ── Tabs ── */
[data-testid="stTabs"] button {
    border-radius: var(--radius-sm) var(--radius-sm) 0 0 !important;
    background: transparent !important;
    border: none !important;
    border-bottom: 2px solid transparent !important;
    font-family: 'Inter', sans-serif !important;
    font-weight: 500 !important;
    padding: 0.7rem 1.2rem !important;
    color: var(--text-muted) !important;
    transition: var(--transition);
    font-size: 0.88rem !important;
}
[data-testid="stTabs"] button:hover {
    color: var(--text-primary) !important;
    background: rgba(0, 212, 255, 0.05) !important;
}
[data-testid="stTabs"] button[aria-selected="true"] {
    color: var(--accent-cyan) !important;
    border-bottom: 2px solid var(--accent-cyan) !important;
    background: rgba(0, 212, 255, 0.08) !important;
}

/* ── Expanders ── */
[data-testid="stExpander"] {
    border: 1px solid var(--border-glass) !important;
    border-radius: var(--radius-sm) !important;
    background: var(--bg-card) !important;
    backdrop-filter: blur(8px);
    transition: var(--transition);
}
[data-testid="stExpander"]:hover {
    border-color: rgba(0, 212, 255, 0.2) !important;
}

/* ── Divider ── */
hr { border-color: var(--border-glass) !important; }

/* ── Metrics ── */
[data-testid="stMetric"] {
    background: var(--bg-card) !important;
    border: 1px solid var(--border-glass) !important;
    border-radius: var(--radius-sm) !important;
    padding: 1rem !important;
    transition: var(--transition);
}
[data-testid="stMetric"]:hover {
    border-color: var(--border-glow) !important;
    box-shadow: 0 0 15px rgba(0, 212, 255, 0.1) !important;
}

/* ── Status widget ── */
[data-testid="stStatusWidget"] {
    background: var(--bg-card) !important;
    border: 1px solid var(--border-glass) !important;
    border-radius: var(--radius-sm) !important;
}

/* ── Selectbox ── */
[data-testid="stSelectbox"] > div > div {
    background: rgba(15, 15, 35, 0.8) !important;
    border: 1px solid var(--border-glass) !important;
    border-radius: var(--radius-sm) !important;
}

/* ── Slider ── */
[data-testid="stSlider"] > div > div > div {
    color: var(--accent-cyan) !important;
}

/* ── Download button ── */
[data-testid="stDownloadButton"] > button {
    background: rgba(34, 197, 94, 0.1) !important;
    border: 1px solid rgba(34, 197, 94, 0.3) !important;
    color: var(--accent-green) !important;
}
[data-testid="stDownloadButton"] > button:hover {
    background: rgba(34, 197, 94, 0.2) !important;
    box-shadow: 0 0 15px rgba(34, 197, 94, 0.2) !important;
}

/* ── Literature Review Glass Container ── */
.lit-review {
    background: var(--bg-card);
    backdrop-filter: blur(12px);
    border: 1px solid var(--border-glass);
    border-radius: var(--radius);
    padding: 2rem;
    animation: fadeInUp 0.5s ease-out;
    line-height: 1.8;
    font-size: 0.95rem;
}
.lit-review h2 { color: var(--accent-cyan) !important; margin-bottom: 1rem; }
.lit-review h3 { color: var(--accent-purple) !important; margin-top: 1.5rem; }
.lit-review li { margin-bottom: 0.4rem; color: var(--text-secondary); }
.lit-review strong { color: var(--text-primary); }
.lit-review hr { border-color: var(--border-glass); margin: 1.5rem 0; }

/* ── Comparison Table ── */
.comparison-table {
    width: 100%;
    border-collapse: separate;
    border-spacing: 0;
    border-radius: var(--radius-sm);
    overflow: hidden;
    border: 1px solid var(--border-glass);
    font-size: 0.88rem;
}
.comparison-table th {
    background: rgba(0, 212, 255, 0.08);
    color: var(--accent-cyan);
    font-weight: 600;
    padding: 0.8rem 1rem;
    text-align: left;
    border-bottom: 1px solid var(--border-glass);
    font-size: 0.8rem;
    text-transform: uppercase;
    letter-spacing: 0.05em;
}
.comparison-table td {
    padding: 0.8rem 1rem;
    border-bottom: 1px solid var(--border-glass);
    color: var(--text-secondary);
    vertical-align: top;
}
.comparison-table tr:hover td {
    background: rgba(0, 212, 255, 0.03);
}
.comparison-table tr:last-child td {
    border-bottom: none;
}

/* ── Sidebar section headers ── */
.sidebar-section {
    font-size: 0.85rem;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 0.1em;
    color: var(--text-muted);
    margin-top: 0.5rem;
    margin-bottom: 0.5rem;
    display: flex;
    align-items: center;
    gap: 0.4rem;
}

/* ── Workflow Stage Status Text ── */
.stage-status {
    font-size: 0.85rem;
    color: var(--text-secondary);
    display: flex;
    align-items: center;
    gap: 0.4rem;
    padding: 0.5rem 0;
    animation: slideInLeft 0.3s ease-out;
}
.stage-status .done { color: var(--accent-green); }
.stage-status .active { color: var(--accent-cyan); }

/* ── Sidebar Logo ── */
.sidebar-logo {
    font-size: 1.3rem;
    font-weight: 800;
    background: linear-gradient(135deg, var(--accent-cyan), var(--accent-purple));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-bottom: 0.2rem;
}
.sidebar-tagline {
    font-size: 0.78rem;
    color: var(--text-muted);
    font-style: italic;
}

/* ── Example question buttons ── */
.example-btn button {
    text-align: left !important;
    font-size: 0.85rem !important;
    color: var(--text-secondary) !important;
    background: transparent !important;
    border: 1px solid transparent !important;
    padding: 0.4rem 0.8rem !important;
}
.example-btn button:hover {
    color: var(--accent-cyan) !important;
    border-color: rgba(0, 212, 255, 0.2) !important;
    background: rgba(0, 212, 255, 0.05) !important;
}

/* ── Spinner override ── */
[data-testid="stSpinner"] {
    color: var(--accent-cyan) !important;
}

/* ── Info/Warning/Success boxes ── */
[data-testid="stAlert"] {
    border-radius: var(--radius-sm) !important;
    backdrop-filter: blur(8px);
}

/* ── Background particle dots (decorative) ── */
.bg-particles {
    position: fixed;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    pointer-events: none;
    z-index: 0;
    overflow: hidden;
}
.particle {
    position: absolute;
    width: 3px;
    height: 3px;
    border-radius: 50%;
    opacity: 0.15;
    animation: float 8s ease-in-out infinite;
}
</style>
""", unsafe_allow_html=True)


# ============================================================
# Background Particle Decoration
# ============================================================
st.markdown("""
<div class="bg-particles">
    <div class="particle" style="background: var(--accent-cyan); top: 15%; left: 10%; animation-delay: 0s; animation-duration: 9s;"></div>
    <div class="particle" style="background: var(--accent-purple); top: 30%; left: 85%; animation-delay: 2s; animation-duration: 7s;"></div>
    <div class="particle" style="background: var(--accent-green); top: 60%; left: 20%; animation-delay: 4s; animation-duration: 11s;"></div>
    <div class="particle" style="background: var(--accent-amber); top: 80%; left: 70%; animation-delay: 1s; animation-duration: 8s;"></div>
    <div class="particle" style="background: var(--accent-rose); top: 45%; left: 50%; animation-delay: 3s; animation-duration: 10s;"></div>
    <div class="particle" style="background: var(--accent-cyan); top: 70%; left: 40%; animation-delay: 5s; animation-duration: 12s;"></div>
    <div class="particle" style="background: var(--accent-purple); top: 20%; left: 60%; animation-delay: 6s; animation-duration: 9s;"></div>
</div>
""", unsafe_allow_html=True)


# ============================================================
# Session State Initialisation
# ============================================================
def init_session():
    defaults = {
        "index_built"        : False,
        "chunks"             : [],
        "sources"            : [],
        "rag_pipeline"       : None,
        "summarizer"         : None,
        "retriever"          : None,
        "embedding_gen"      : None,
        "vector_db"          : None,
        "qa_history"         : [],       # Chat history for Q&A tab
        "bookmarked_papers"  : [],       # Bookmarked papers from agentic workflow
        "last_workflow_state": None,     # Last agentic workflow result for PDF export
        "last_workflow_topic": "",       # Last agentic workflow topic
    }
    for key, val in defaults.items():
        if key not in st.session_state:
            st.session_state[key] = val

init_session()


# ============================================================
# Cached Resource Loaders (load once per session)
# ============================================================
@st.cache_resource(show_spinner="Loading embedding model…")
def load_embedding_gen():
    return EmbeddingGenerator()

@st.cache_resource(show_spinner="Loading QA model…")
def load_rag_pipeline():
    return RAGPipeline(retriever=None)

@st.cache_resource(show_spinner="Loading summarizer…")
def load_summarizer():
    return Summarizer()


# ============================================================
# Helper: Confidence Badge HTML
# ============================================================
def confidence_badge(score: float) -> str:
    return '<span class="badge badge-synth">🤖 Synthesized</span>'


# ============================================================
# Helper: Render Pipeline Progress HTML
# ============================================================
PIPELINE_STAGES = [
    ("📋", "Plan"),
    ("🔍", "Search"),
    ("📊", "Rank"),
    ("🧠", "Analyze"),
    ("⚖️", "Compare"),
    ("📝", "Review"),
]

def render_pipeline(current_stage_idx: int, total_stages: int = 6) -> str:
    """
    Generates the HTML for the pipeline progress indicator.
    current_stage_idx: 0-based index of the currently active stage.
                       -1 = all pending, total_stages = all done.
    """
    html = '<div class="pipeline-container">'
    for i, (icon, label) in enumerate(PIPELINE_STAGES):
        if i < current_stage_idx:
            icon_class = "done"
            label_class = "done"
            display_icon = "✓"
        elif i == current_stage_idx:
            icon_class = "active"
            label_class = "active"
            display_icon = icon
        else:
            icon_class = "pending"
            label_class = ""
            display_icon = icon

        html += f'''
        <div class="pipeline-node">
            <div class="pipeline-icon {icon_class}">{display_icon}</div>
            <div class="pipeline-label {label_class}">{label}</div>
        </div>
        '''
        # Connector (not after the last node)
        if i < total_stages - 1:
            if i < current_stage_idx:
                conn_class = "done"
            elif i == current_stage_idx:
                conn_class = "active"
            else:
                conn_class = ""
            html += f'<div class="pipeline-connector {conn_class}"></div>'

    html += '</div>'
    return html


# ============================================================
# Helper: Format Insight Block (Glass Card)
# ============================================================
def render_insight(title: str, content, css_class: str, icon: str):
    if isinstance(content, list):
        items_html = "".join(f"<li>{item}</li>" for item in content if item)
        body = f"<ul>{items_html}</ul>"
    else:
        body = f"<p style='color: var(--text-secondary); margin: 0;'>{content}</p>"

    st.markdown(f"""
    <div class="insight-card {css_class}">
        <h4>{icon} {title}</h4>
        {body}
    </div>
    """, unsafe_allow_html=True)


# ============================================================
# Helper: Render Paper Result Card
# ============================================================
def render_paper_card(paper: dict, rank: int) -> str:
    title = paper.get('title', 'Unknown Title')
    year = paper.get('year', 'N/A')
    score = paper.get('relevance_score', 0)
    score_pct = min(score * 100, 100)

    rank_class = {1: "rank-1", 2: "rank-2", 3: "rank-3"}.get(rank, "rank-default")

    return f"""
    <div class="result-paper" style="animation-delay: {rank * 0.08}s;">
        <div class="paper-rank {rank_class}">{rank}</div>
        <div class="paper-info">
            <div class="paper-title">{title}</div>
            <div class="paper-meta">
                <span>📅 {year}</span>
                <span>Score: {score:.3f}</span>
                <div class="paper-score-bar">
                    <div class="paper-score-fill" style="width: {score_pct}%;"></div>
                </div>
            </div>
        </div>
    </div>
    """


# ============================================================
# Helper: Build Comparison Table HTML
# ============================================================
def build_comparison_html(analyses: list) -> str:
    if not analyses:
        return "<p>No papers to compare.</p>"

    html = '<table class="comparison-table">'
    html += """<thead><tr>
        <th>Paper</th>
        <th>Proposed Method</th>
        <th>Dataset</th>
        <th>Key Findings</th>
        <th>Limitations</th>
    </tr></thead><tbody>"""

    for p in analyses:
        title = p['title']
        method = str(p.get('proposed_method', '')).replace('\n', ' ')[:180]
        dataset = str(p.get('dataset', '')).replace('\n', ' ')[:120]

        findings = p.get('key_findings', '')
        if isinstance(findings, list):
            findings = findings[0] if findings else "—"
        findings = str(findings).replace('\n', ' ')[:180]

        limitations = p.get('limitations', '')
        if isinstance(limitations, list):
            limitations = limitations[0] if limitations else "—"
        limitations = str(limitations).replace('\n', ' ')[:120]

        html += f"""<tr>
            <td><strong>{title}</strong></td>
            <td>{method}</td>
            <td>{dataset}</td>
            <td>{findings}</td>
            <td>{limitations}</td>
        </tr>"""

    html += "</tbody></table>"
    return html


# ============================================================
# Helper: Safe markdown-to-HTML (for literature review)
# ============================================================
def _md_to_safe_html(md_text: str) -> str:
    """
    Converts basic markdown to HTML for display in the lit-review div.
    Handles headers, bold, lists, horizontal rules.
    """
    import re

    lines = md_text.split('\n')
    html_lines = []
    in_list = False

    for line in lines:
        stripped = line.strip()

        # Horizontal rule
        if stripped.startswith('---'):
            if in_list:
                html_lines.append('</ul>')
                in_list = False
            html_lines.append('<hr/>')
            continue

        # Headers
        if stripped.startswith('### '):
            if in_list:
                html_lines.append('</ul>')
                in_list = False
            html_lines.append(f'<h3>{stripped[4:]}</h3>')
            continue
        if stripped.startswith('## '):
            if in_list:
                html_lines.append('</ul>')
                in_list = False
            html_lines.append(f'<h2>{stripped[3:]}</h2>')
            continue

        # List items
        if stripped.startswith('- '):
            if not in_list:
                html_lines.append('<ul>')
                in_list = True
            content = stripped[2:]
            # Handle bold
            content = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', content)
            html_lines.append(f'<li>{content}</li>')
            continue

        # Close list if we're in one
        if in_list and not stripped.startswith('- '):
            html_lines.append('</ul>')
            in_list = False

        # Regular paragraph
        if stripped:
            # Handle bold
            content = re.sub(r'\*\*(.+?)\*\*', r'<strong>\1</strong>', stripped)
            # Handle italic
            content = re.sub(r'\*(.+?)\*', r'<em>\1</em>', content)
            html_lines.append(f'<p>{content}</p>')

    if in_list:
        html_lines.append('</ul>')

    return '\n'.join(html_lines)


# ============================================================
# SIDEBAR — Premium Navigation
# ============================================================
with st.sidebar:
    st.markdown('<div class="sidebar-logo">🧠 Research Engine</div>', unsafe_allow_html=True)
    st.markdown('<div class="sidebar-tagline">Powered by RAG + HuggingFace</div>', unsafe_allow_html=True)
    st.divider()

    # ── Upload Section ──
    st.markdown('<div class="sidebar-section">📂 Upload Papers</div>', unsafe_allow_html=True)
    uploaded_files = st.file_uploader(
        label       = "Drop PDF files here",
        type        = ["pdf"],
        accept_multiple_files = True,
        key         = "pdf_uploader",
        help        = "Upload one or more research paper PDFs.",
    )

    # ── Chunking settings ──
    st.divider()
    st.markdown('<div class="sidebar-section">⚙️ Settings</div>', unsafe_allow_html=True)
    chunk_size    = st.slider("Chunk Size (chars)",    200, 1000, 500, 50,
                              help="Size of each text chunk passed to the embedder.")
    chunk_overlap = st.slider("Chunk Overlap (chars)",  0,  200, 100, 10,
                              help="Overlap between consecutive chunks.")
    top_k         = st.slider("Top-K Retrieval",        1,   10,   5,  1,
                              help="Number of context chunks retrieved per query.")

    # ── Process button ──
    st.divider()
    process_btn = st.button("🚀 Process & Index PDFs", use_container_width=True)

    # ── Index stats ──
    if st.session_state.index_built:
        st.divider()
        st.markdown('<div class="sidebar-section">📊 Index Stats</div>', unsafe_allow_html=True)
        vdb   = st.session_state.vector_db
        stats = vdb.get_stats() if vdb else {}
        st.metric("Total Chunks",   stats.get("total_chunks",  0))
        st.metric("Total Vectors",  stats.get("total_vectors", 0))
        sources = stats.get("sources", [])
        st.markdown(f"**Papers indexed:** {len(sources)}")
        for s in sources:
            st.markdown(f'<span class="source-chip">📄 {s}</span>', unsafe_allow_html=True)

    # ── Bookmarked Papers ──
    if st.session_state.bookmarked_papers:
        st.divider()
        st.markdown('<div class="sidebar-section">⭐ Bookmarked Papers</div>', unsafe_allow_html=True)
        for bm in st.session_state.bookmarked_papers:
            st.markdown(f"""
            <div class="glass-card" style="padding: 0.6rem; margin-bottom: 0.4rem; font-size: 0.8rem;">
                <strong>{bm.get('title', 'Unknown')[:50]}</strong><br/>
                <span style="color: var(--text-secondary);">{bm.get('year', 'N/A')} · Score: {bm.get('relevance_score', 0):.2f}</span>
            </div>
            """, unsafe_allow_html=True)

    # ── Footer ──
    st.divider()
    st.caption("Agentic AI Research Scientist System")
    st.caption("Built with ❤️ using Streamlit + LangChain + FAISS")


# ============================================================
# MAIN — Animated Header
# ============================================================
st.markdown("""
<div class="app-header">
    <h1>🧠 Research Paper Intelligence Engine</h1>
    <div class="app-subtitle">
        Upload research papers → ask questions → get AI-powered answers, summaries, and structured insights.
    </div>
    <div class="header-decoration">
        <div class="header-dot"></div>
        <div class="header-dot"></div>
        <div class="header-dot"></div>
        <div class="header-dot"></div>
        <div class="header-dot"></div>
    </div>
</div>
""", unsafe_allow_html=True)


# ============================================================
# PDF Processing Logic (triggered by button)
# ============================================================
if process_btn:
    if not uploaded_files:
        st.warning("⚠️ Please upload at least one PDF before processing.")
    else:
        with st.status("📥 Processing PDFs…", expanded=True) as status:
            try:
                processor   = PDFProcessor()
                chunker     = TextChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
                embed_gen   = load_embedding_gen()
                vdb         = VectorDB()

                all_chunks  = []
                docs        = {}

                # Step 1: Extract text from each PDF
                for i, uf in enumerate(uploaded_files):
                    st.write(f"📄 Extracting: **{uf.name}**")

                    # Extract text directly from memory buffer
                    result = processor.process_single(stream=uf.getvalue(), filename=uf.name)
                    docs[uf.name] = result

                # Step 2: Chunk all documents
                st.write("✂️ Chunking text…")
                all_chunks = chunker.chunk_documents(docs)

                # Step 3: Generate embeddings
                st.write(f"🔢 Generating embeddings for {len(all_chunks)} chunks…")
                embeddings = embed_gen.embed_chunks(all_chunks)

                # Step 4: Build FAISS index
                st.write("🗂️ Building FAISS index…")
                vdb.build_index(embeddings, all_chunks)
                vdb.save()

                # Save to session state
                retriever = Retriever(embedding_gen=embed_gen, vector_db=vdb)
                st.session_state.index_built   = True
                st.session_state.chunks        = all_chunks
                st.session_state.sources       = list(docs.keys())
                st.session_state.vector_db     = vdb
                st.session_state.embedding_gen = embed_gen
                st.session_state.retriever     = retriever

                status.update(
                    label=f"✅ Indexed {len(all_chunks)} chunks from {len(docs)} PDF(s)!",
                    state="complete",
                )
            except Exception as exc:
                status.update(label=f"❌ Error: {exc}", state="error")
                logger.exception(exc)


# ============================================================
# TABS: Agentic | Q&A | Summarize | Insights | Context
# ============================================================
tab_agent, tab_qa, tab_summary, tab_insights, tab_context = st.tabs([
    "🤖 Agentic Workflow",
    "💬 Ask a Question",
    "📝 Summarize Paper",
    "🔍 Research Insights",
    "📚 Retrieved Context",
])

# ──────────────────────────────────────────────────────────
# TAB 0: Agentic Workflow — with Pipeline Progress
# ──────────────────────────────────────────────────────────
with tab_agent:
    st.markdown("""
    <div class="glass-card" style="text-align:center; padding: 1.2rem;">
        <h2 style="margin:0;">🤖 Agentic AI Research Assistant</h2>
        <p style="color: var(--text-secondary); margin: 0.3rem 0 0;">
            End-to-end automated research workflow: Discover → Rank → Analyze → Compare → Review
        </p>
    </div>
    """, unsafe_allow_html=True)

    col_topic, col_limit = st.columns([3, 1])
    with col_topic:
        topic = st.text_input(
            "Research Topic",
            placeholder="e.g., Retrieval-Augmented Generation in Healthcare",
            key="agent_topic"
        )
    with col_limit:
        num_papers = st.slider("Number of Papers", 1, 10, 5, key="num_papers_slider")

    agent_btn = st.button("🚀 Run Full Research Workflow", use_container_width=True)

    if agent_btn and topic:
        if not topic.strip():
            st.error("⚠️ Please enter a valid research topic before starting.")
        else:
            # Show initial pipeline (all pending)
            pipeline_placeholder = st.empty()
            pipeline_placeholder.markdown(render_pipeline(-1), unsafe_allow_html=True)

            stage_status = st.empty()

            try:
                from src.agent import (
                    plan_research_node, search_papers_node, rank_papers_node,
                    analyze_papers_node, compare_papers_node,
                    review_papers_node, ResearchState
                )

                state = ResearchState(
                    research_topic=topic,
                    topic=topic,
                    research_questions=[],
                    discovered_papers=[],
                    raw_papers=[],
                    ranked_papers=[],
                    selected_papers=[],
                    paper_analysis=[],
                    analyses=[],
                    comparison="",
                    literature_review="",
                    status="Starting...",
                    errors=[]
                )

                # ── Stage 0: Plan ──
                pipeline_placeholder.markdown(render_pipeline(0), unsafe_allow_html=True)
                stage_status.markdown(
                    '<div class="stage-status"><span class="active">⟳</span> Formulating research plan & scope…</div>',
                    unsafe_allow_html=True
                )
                state = plan_research_node(state)

                # ── Stage 1: Search ──
                pipeline_placeholder.markdown(render_pipeline(1), unsafe_allow_html=True)
                stage_status.markdown(
                    '<div class="stage-status"><span class="active">⟳</span> Searching arXiv for papers on this topic…</div>',
                    unsafe_allow_html=True
                )
                state = search_papers_node(state)
                num_found = len(state.get("discovered_papers", []))

                if num_found == 0:
                    st.warning(f"⚠️ No papers found on arXiv for query '{topic}'. Try broadening your search terms.")
                else:
                    # ── Stage 2: Rank ──
                    pipeline_placeholder.markdown(render_pipeline(2), unsafe_allow_html=True)
                    stage_status.markdown(
                        f'<div class="stage-status"><span class="done">✓</span> Discovered {num_found} papers — <span class="active">⟳</span> Ranking top-{num_papers} by semantic relevance…</div>',
                        unsafe_allow_html=True
                    )
                    # Override Top-K selection with UI slider
                    from src.ranker import SemanticRanker
                    ranker = SemanticRanker(top_k=num_papers)
                    state["ranked_papers"] = ranker.rank_papers(topic, state.get("discovered_papers", []))
                    state["selected_papers"] = state["ranked_papers"]
                    num_ranked = len(state.get("selected_papers", []))

                    # ── Stage 3: Analyze ──
                    pipeline_placeholder.markdown(render_pipeline(3), unsafe_allow_html=True)
                    stage_status.markdown(
                        f'<div class="stage-status"><span class="done">✓</span> Selected top {num_ranked} papers — <span class="active">⟳</span> Ingesting into VT RAG engine & extracting insights…</div>',
                        unsafe_allow_html=True
                    )
                    state = analyze_papers_node(state)

                    # ── Stage 4: Compare ──
                    pipeline_placeholder.markdown(render_pipeline(4), unsafe_allow_html=True)
                    stage_status.markdown(
                        '<div class="stage-status"><span class="done">✓</span> Analysis complete — <span class="active">⟳</span> Generating comparative matrix…</div>',
                        unsafe_allow_html=True
                    )
                    state = compare_papers_node(state)

                    # ── Stage 5: Review ──
                    pipeline_placeholder.markdown(render_pipeline(5), unsafe_allow_html=True)
                    stage_status.markdown(
                        '<div class="stage-status"><span class="done">✓</span> Matrix ready — <span class="active">⟳</span> Synthesizing literature review…</div>',
                        unsafe_allow_html=True
                    )
                    state = review_papers_node(state)

                    # ── All Done ──
                    pipeline_placeholder.markdown(render_pipeline(6), unsafe_allow_html=True)
                    stage_status.markdown(
                        '<div class="stage-status"><span class="done">✓</span> Research Workflow complete! All 6 stages finished successfully.</div>',
                        unsafe_allow_html=True
                    )

                    # ════════════════════════════════════════════════
                    # Display Results
                    # ════════════════════════════════════════════════

                    st.markdown("---")

                    # ── Research Plan Questions ──
                    st.markdown("### 📋 Formulated Research Plan & Scope")
                    for q in state.get("research_questions", []):
                        st.markdown(f"- ❓ {q}")

                    st.markdown("---")

                    # ── Top Ranked Papers ──
                    st.markdown("### 🏆 Top Ranked Papers")
                    papers_html = ""
                    for i, p in enumerate(state.get("selected_papers", []), start=1):
                        papers_html += render_paper_card(p, i)
                    st.markdown(papers_html, unsafe_allow_html=True)

                    st.markdown("---")

                    # ── Structured Analysis ──
                    st.markdown("### 📊 Structured Analysis")
                    for idx, p_analysis in enumerate(state.get("analyses", [])):
                        with st.expander(f"📄 {p_analysis['title']}", expanded=(idx == 0)):
                            fields = [
                                ("🎯 Research Problem", p_analysis.get("research_problem", "")),
                                ("🎯 Objective", p_analysis.get("objective", "")),
                                ("🔬 Proposed Method", p_analysis.get("proposed_method", "")),
                                ("📊 Dataset", p_analysis.get("dataset", "")),
                                ("📏 Evaluation Metrics", p_analysis.get("evaluation_metrics", "")),
                                ("📈 Results", p_analysis.get("experimental_results", "")),
                            ]
                            for label, value in fields:
                                st.markdown(f"**{label}**")
                                st.markdown(f"> {value}")

                            col1, col2, col3 = st.columns(3)
                            with col1:
                                render_insight("Key Findings", p_analysis.get("key_findings", "—"), "findings", "🟢")
                            with col2:
                                render_insight("Limitations", p_analysis.get("limitations", "—"), "limits", "🟡")
                            with col3:
                                render_insight("Future Work", p_analysis.get("future_work", "—"), "future", "🔵")

                    st.markdown("---")

                    # ── Multi-Paper Comparison ──
                    st.markdown("### ⚖️ Multi-Paper Comparison")
                    comparison_html = build_comparison_html(state.get("analyses", []))
                    st.markdown(f'<div class="glass-card" style="padding: 0; overflow-x: auto;">{comparison_html}</div>', unsafe_allow_html=True)

                    st.markdown("---")

                    # ── Literature Review ──
                    lit_review = state.get("literature_review", "")
                    if lit_review:
                        st.markdown(f'<div class="lit-review">{_md_to_safe_html(lit_review)}</div>', unsafe_allow_html=True)

                    # ── Save state for export ──
                    st.session_state.last_workflow_state = state
                    st.session_state.last_workflow_topic = topic

                    st.markdown("---")

                    # ── Action buttons row ──
                    action_col1, action_col2, action_col3 = st.columns([1, 1, 2])

                    with action_col1:
                        try:
                            from src.export import ResearchReportPDF
                            exporter = ResearchReportPDF()
                            pdf_bytes = exporter.generate(topic, state)
                            st.download_button(
                                label="📥 Download Report (PDF)",
                                data=pdf_bytes,
                                file_name=f"research_report_{topic.replace(' ', '_')[:30]}.pdf",
                                mime="application/pdf",
                                key="download_pdf_report",
                                use_container_width=True,
                            )
                        except ImportError:
                            st.warning("PDF export requires fpdf2. Install: pip install fpdf2")
                        except Exception as pdf_exc:
                            st.error(f"PDF export failed: {pdf_exc}")

                    with action_col2:
                        if st.button("⭐ Bookmark All Papers", key="bookmark_all", use_container_width=True):
                            existing_ids = {p.get('id') for p in st.session_state.bookmarked_papers}
                            for p in state.get("selected_papers", []):
                                if p.get('id') not in existing_ids:
                                    st.session_state.bookmarked_papers.append(p)
                            st.toast(f"⭐ Bookmarked {len(state.get('selected_papers', []))} papers!")
                            st.rerun()

            except Exception as exc:
                pipeline_placeholder.markdown(render_pipeline(-1), unsafe_allow_html=True)
                stage_status.empty()
                st.error(f"❌ Workflow Error: {exc}")
                logger.exception(exc)

    elif agent_btn and not topic:
        st.warning("Please enter a research topic first.")

    # ── Re-export from last run ──
    elif st.session_state.last_workflow_state is not None:
        st.markdown("---")
        st.info("Previous workflow results are available. Run a new workflow or download the last report.")
        try:
            from src.export import ResearchReportPDF
            exporter = ResearchReportPDF()
            pdf_bytes = exporter.generate(
                st.session_state.last_workflow_topic,
                st.session_state.last_workflow_state,
            )
            st.download_button(
                label="📥 Download Last Report (PDF)",
                data=pdf_bytes,
                file_name=f"research_report_{st.session_state.last_workflow_topic.replace(' ', '_')[:30]}.pdf",
                mime="application/pdf",
                key="download_pdf_last",
            )
        except Exception:
            pass


# ──────────────────────────────────────────────────────────
# TAB 1: Q&A via RAG
# ──────────────────────────────────────────────────────────
with tab_qa:
    st.markdown("""
    <div class="glass-card" style="text-align:center; padding: 1.2rem;">
        <h2 style="margin:0;">💬 Ask Questions About Your Papers</h2>
        <p style="color: var(--text-secondary); margin: 0.3rem 0 0;">
            Ask anything — the AI retrieves relevant context and answers from your papers.
        </p>
    </div>
    """, unsafe_allow_html=True)

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first using the sidebar.")
    else:
        question = st.text_input(
            label       = "Your question",
            placeholder = "What is the main contribution of this paper?",
            key         = "qa_question",
        )

        col1, col2 = st.columns([1, 5])
        with col1:
            ask_btn = st.button("🔎 Ask", key="ask_btn", use_container_width=True)

        if ask_btn and question:
            try:
                rag = load_rag_pipeline()
                rag.retriever = st.session_state.retriever

                st.markdown(f'### Answer {confidence_badge(1.0)}', unsafe_allow_html=True)
                answer_placeholder = st.empty()

                with st.spinner("🤔 Searching papers & thinking…"):
                    result = rag.answer(question, top_k=top_k, stream=True)

                streamer = result["streamer"]
                thread = result["thread"]

                full_answer = ""
                for new_text in streamer:
                    full_answer += new_text
                    # Display with animated cursor
                    answer_placeholder.markdown(
                        f'<div class="answer-box streaming">{full_answer}<span class="cursor-blink">▌</span></div>',
                        unsafe_allow_html=True,
                    )

                # Remove cursor when done
                answer_placeholder.markdown(
                    f'<div class="answer-box">{full_answer}</div>',
                    unsafe_allow_html=True,
                )
                thread.join()

                # ── Sources ──
                st.markdown("**Sources used:**")
                unique_sources = {s["source"] for s in result["sources"]}
                chips = " ".join(
                    f'<span class="source-chip">📄 {s}</span>'
                    for s in unique_sources
                )
                st.markdown(chips, unsafe_allow_html=True)

                # ── Save to chat history ──
                st.session_state.qa_history.append({
                    "question": question,
                    "answer": full_answer,
                    "sources": list(unique_sources),
                })

                # ── Expandable context ──
                with st.expander("🔎 View retrieved context"):
                    st.code(result["context"][:2000], language=None)

            except Exception as exc:
                st.error(f"❌ Error: {exc}")
                logger.exception(exc)

        elif ask_btn and not question:
            st.warning("Please type a question first.")

        # ── Example questions ──
        st.divider()
        st.markdown("**💡 Example questions:**")
        examples = [
            "What is the main contribution of this paper?",
            "What dataset was used for evaluation?",
            "What are the experimental results?",
            "What deep learning architecture is proposed?",
            "What problem does this paper solve?",
        ]

        def set_q(q):
            st.session_state.qa_question = q

        for ex in examples:
            st.button(f"▷ {ex}", key=f"ex_{ex[:20]}", on_click=set_q, args=(ex,))

        # ── Chat History ──
        if st.session_state.qa_history:
            st.divider()
            col_hist, col_clear = st.columns([4, 1])
            with col_hist:
                st.markdown("### 💬 Chat History")
            with col_clear:
                if st.button("🗑️ Clear", key="clear_history"):
                    st.session_state.qa_history = []
                    st.rerun()

            for i, entry in enumerate(reversed(st.session_state.qa_history)):
                idx = len(st.session_state.qa_history) - i
                st.markdown(f"""
                <div class="glass-card" style="padding: 0.8rem; margin-bottom: 0.5rem;">
                    <div style="color: var(--accent-cyan); font-weight: 600; margin-bottom: 0.3rem;">❓ Q{idx}: {entry['question']}</div>
                    <div style="color: var(--text-primary); margin-bottom: 0.3rem;">{entry['answer']}</div>
                    <div style="font-size: 0.75rem; color: var(--text-secondary);">Sources: {', '.join(entry.get('sources', []))}</div>
                </div>
                """, unsafe_allow_html=True)


# ──────────────────────────────────────────────────────────
# TAB 2: Summarization
# ──────────────────────────────────────────────────────────
with tab_summary:
    st.markdown("""
    <div class="glass-card" style="text-align:center; padding: 1.2rem;">
        <h2 style="margin:0;">📝 Paper Summarization</h2>
        <p style="color: var(--text-secondary); margin: 0.3rem 0 0;">
            Generate a concise abstract-style summary of any indexed paper.
        </p>
    </div>
    """, unsafe_allow_html=True)

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first.")
    else:
        sources = st.session_state.sources
        selected = st.selectbox(
            "Select a paper to summarize",
            options=["All Papers"] + sources,
            key="summary_source",
        )

        sum_btn = st.button("📝 Generate Summary", key="sum_btn")

        if sum_btn:
            with st.spinner("🤔 Generating summary with BART…"):
                try:
                    summ = load_summarizer()
                    chunks = st.session_state.chunks

                    if selected == "All Papers":
                        full_text = "\n\n".join(c["text"] for c in chunks)
                        src_label = "All Papers"
                    else:
                        full_text = "\n\n".join(
                            c["text"] for c in chunks if c.get("source") == selected
                        )
                        src_label = selected

                    summary = summ.summarize(full_text)

                    st.markdown(f"### Summary: *{src_label}*")
                    st.markdown(
                        f'<div class="glass-card"><p style="line-height:1.8; color: var(--text-secondary);">{summary}</p></div>',
                        unsafe_allow_html=True,
                    )
                    st.download_button(
                        "⬇️ Download Summary",
                        data=summary,
                        file_name=f"summary_{src_label.replace(' ','_')}.txt",
                        mime="text/plain",
                        key="download_summary",
                    )
                except Exception as exc:
                    st.error(f"❌ Error: {exc}")
                    logger.exception(exc)


# ──────────────────────────────────────────────────────────
# TAB 3: Research Insights
# ──────────────────────────────────────────────────────────
with tab_insights:
    st.markdown("""
    <div class="glass-card" style="text-align:center; padding: 1.2rem;">
        <h2 style="margin:0;">🔍 Research Insights Extraction</h2>
        <p style="color: var(--text-secondary); margin: 0.3rem 0 0;">
            Automatically extract <strong>Key Findings</strong>, <strong>Limitations</strong>, and <strong>Future Work</strong> from your research paper.
        </p>
    </div>
    """, unsafe_allow_html=True)

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first.")
    else:
        sources  = st.session_state.sources
        selected = st.selectbox(
            "Select a paper",
            options=sources,
            key="insights_source",
        )

        ins_btn = st.button("🔍 Extract Insights", key="ins_btn")

        if ins_btn:
            with st.spinner("🕵️ Extracting structured insights…"):
                try:
                    summ   = load_summarizer()
                    chunks = st.session_state.chunks
                    result = summ.full_analysis(chunks, source=selected)

                    # ── Summary card ──
                    st.markdown(f"### 📄 {selected}")
                    st.markdown(
                        f'<div class="glass-card"><strong>Summary:</strong><p style="color: var(--text-secondary); line-height: 1.7;">{result["summary"]}</p></div>',
                        unsafe_allow_html=True,
                    )

                    # ── Insight cards ──
                    col1, col2, col3 = st.columns(3)

                    with col1:
                        render_insight(
                            "Key Findings",
                            result["key_findings"],
                            "findings",
                            "🟢",
                        )
                    with col2:
                        render_insight(
                            "Limitations",
                            result["limitations"],
                            "limits",
                            "🟡",
                        )
                    with col3:
                        render_insight(
                            "Future Work",
                            result["future_work"],
                            "future",
                            "🔵",
                        )

                    # ── Download ──
                    import json
                    report = json.dumps(result, indent=2, ensure_ascii=False)
                    st.download_button(
                        "⬇️ Download Insights (JSON)",
                        data=report,
                        file_name=f"insights_{selected.replace(' ','_')}.json",
                        mime="application/json",
                        key="download_insights",
                    )

                except Exception as exc:
                    st.error(f"❌ Error: {exc}")
                    logger.exception(exc)


# ──────────────────────────────────────────────────────────
# TAB 4: Retrieved Context Explorer
# ──────────────────────────────────────────────────────────
with tab_context:
    st.markdown("""
    <div class="glass-card" style="text-align:center; padding: 1.2rem;">
        <h2 style="margin:0;">📚 Semantic Search Explorer</h2>
        <p style="color: var(--text-secondary); margin: 0.3rem 0 0;">
            Search the indexed chunks directly to see what's retrieved for any query — useful for debugging and understanding the pipeline.
        </p>
    </div>
    """, unsafe_allow_html=True)

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first.")
    else:
        search_query = st.text_input(
            "Search query",
            placeholder="attention mechanism transformer",
            key="ctx_query",
        )
        search_btn = st.button("🔎 Search Chunks", key="ctx_btn")

        if search_btn and search_query:
            with st.spinner("Searching…"):
                try:
                    retriever = st.session_state.retriever
                    results   = retriever.retrieve(search_query, top_k=top_k)

                    st.markdown(f"**Found {len(results)} chunks:**")
                    for r in results:
                        with st.expander(
                            f"Rank {r['rank']} | Score: {r['score']:.4f} | 📄 {r['source']}"
                        ):
                            st.text(r["text"])

                except Exception as exc:
                    st.error(f"❌ Error: {exc}")


### Install Dependencies

In [ ]:
!pip install -r requirements.txt
!npm install localtunnel

### Get your Endpoint IP
Copy the IP address below. You will need to paste it into the localtunnel website.

In [ ]:
import urllib
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

### Launch App
Click the `loca.lt` link below and paste the IP address!

In [ ]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501